# 01. 데이터 불러오기, 점검, 저장

## 이 노트북이 하는 일
`dataset/` 폴더의 원천 데이터 8종을 **읽고 -> 형식을 맞추고 -> 문제가 없는지 점검하고 -> 분석하기 좋은 파일로 저장**한다.

| 원천 데이터 (`dataset/`) | 저장 결과 (`notebooks/preprocessed/`) |
|---|---|
| 01 카드/유동인구 zip | `card1`, `card2`, `flow_age`, `flow_time`, `flow_wkdy`, `industry_map.csv` |
| 02 기상관측 zip | `wx_daily`, `wx_hourly` |
| 03 기상특보 csv | `wx_warning_raw` |
| 04 상권매출 zip | `sales_seoul` |
| 05 지하철 zip | `subway` |
| 06 상가정보 zip | `stores` |
| 07 관광방문자 zip | `tour_dom_trend`, `tour_dom_dong`, `tour_dom_home`, `tour_for_trend`, `tour_for_dong`, `tour_for_country` |
| 08 행정동 경계 | `dong_boundary.geojson`, `dong_codes.csv` |
| (직접 만듦) | `calendar` |

## 실행 방법
- 커널은 **`26bigcontest`** 를 선택한다. (VS Code 오른쪽 위 커널 선택)
- 위에서부터 차례로 실행한다. `Shift + Enter` = 지금 셀을 실행하고 다음 셀로 이동
- 한 번에 전부 실행하려면 위쪽의 **Run All**. 몇 분 정도 걸린다.
- 저장 폴더(`notebooks/preprocessed/`)는 `.gitignore`에 들어 있어서 GitHub에 올라가지 않는다. 팀원은 각자 이 노트북을 실행해서 만들면 된다.

## 결과 요약
- **카드 데이터1, 2**: 184일 빠짐없이 있고 중복도 없다.
- **5건 미만 마스킹**: 결제가 5건 미만인 칸은 삭제돼 있어서 일별 행 수가 4,268~6,023개로 달라짐. 184일 내내 나오는 칸(`is_balanced`)은 분석 대상 행의 41%이지만 **금액으로는 82%**
- **날씨와 무관하거나 정체를 알 수 없는 업종 23개**(분류 불명인 ZZ_나머지와 체인점, 세금공과금 등)가 데이터1 금액의 **63%** 를 차지해서 분석 대상에서 뺌(`is_target`)
- **SK 유동인구**: 12월 시간대별, 요일별 파일은 모든 행이 두 번씩 들어 있어서 중복을 지움. 월 단위로 집계되어있음.
- **기상, 지하철**: 2025년 하반기 일자료는 모든 지점에 184일이 다 있다. 지하철은 강남 32개, 춘천 5개 노선별 역이 184일 모두 있다. 다만, 신분당선은 목록에도 없고 집계에도 빠져있음. (ex. 2호선 강남역 승하차는 잡히는데 신분당선 강남역 승하차는 안잡힘)
- **행정동**: 일원2동 -> 개포3동 코드 변경을 반영
- 발견한 문제는 맨 아래 **13-2 표**와 `data_issues.csv`에 모아 두었다.

## 0. 준비

### 0-1. 라이브러리 불러오기

In [1]:
import io            # 메모리 안의 데이터를 파일처럼 다루는 도구 (zip 안에 든 zip을 열 때 사용)
import re            # 정규식: 글자 패턴 찾기 (예: 파일 이름에서 연도 뽑기)
import zipfile       # zip을 풀지 않고 안의 파일을 바로 읽는 도구
import warnings
from pathlib import Path   # 파일 경로를 다루는 도구 (윈도우/맥 상관없이 동작)
from IPython.display import display   # 표를 칸이 나뉜 표(HTML)로 보여줌. print(표)는 글자로만 나와서 보기 불편함

# geopandas를 불러올 때 뜨는 GDAL 경고는 동작에 영향이 없어서 숨길다. (import보다 먼저 설정해야 함)
warnings.filterwarnings('ignore', message='.*GDAL.*')

import numpy as np          # 숫자 계산
import pandas as pd         # 표(DataFrame) 다루기 — 이 노트북의 주인공
import geopandas as gpd     # 지도 데이터(행정동 경계) 다루기

# ── 출력 모양 설정 ──
pd.set_option('display.max_columns', 60)    # 표를 출력할 때 열을 최대 60개까지 보여줌 (기본값은 중간을 ... 으로 생략)
pd.set_option('display.width', 200)         # 한 줄에 200글자까지 출력

print('pandas', pd.__version__, '| geopandas', gpd.__version__)

pandas 2.2.3 | geopandas 1.1.3


### 0-2. 경로와 기본값
파일 경로와 분석 기간을 **한곳에** 모아둔다. 나중에 파일 이름이 바뀌면 여기만 고치면 된다.

In [2]:
# 노트북은 notebooks/ 폴더에 있고, 데이터는 한 단계 위의 dataset/ 폴더에 있다.
# 어디서 실행하든 되도록, 'dataset 폴더가 있는 곳'이 나올 때까지 상위 폴더로 올라가며 찾음
ROOT = Path.cwd()
while not (ROOT / 'dataset').exists():
    if ROOT.parent == ROOT:    # 드라이브 맨 위까지 올라갔는데도 없으면 멈춤
        raise FileNotFoundError('dataset 폴더를 찾을 수 없다. 노트북 위치를 확인한다.')
    ROOT = ROOT.parent

DATA = ROOT / 'dataset'                       # 원본 데이터 폴더
OUT = ROOT / 'notebooks' / 'preprocessed'     # 정리한 결과를 저장할 폴더
OUT.mkdir(parents=True, exist_ok=True)        # 폴더가 없으면 새로 만듦 (있으면 그냥 넘어감)

# 원본 파일 경로 목록
SRC = {
    'card':     DATA / '01_대회제공_카드유동인구_202507-202512.zip',
    'weather':  DATA / '02_기상관측_ASOS_AWS_197507-202512.zip',
    'warning':  DATA / '03_기상특보_202507-202512.csv',
    'sales':    DATA / '04_상권매출_행정동_202101-202512.zip',
    'subway':   DATA / '05_지하철승하차_역별_202301-202608.zip',
    'stores':   DATA / '06_상가정보_강남춘천_202510_202512.zip',
    'tour':     DATA / '07_관광방문자_강남춘천_202101-202512.zip',
    'boundary': DATA / '08_행정동경계_전국_20250401' / 'HangJeongDong_ver20250401.geojson',
}

# 파일이 전부 있는지 먼저 확인. 없는 파일이 있으면 stop
for key, path in SRC.items():
    print(f"{'OK  ' if path.exists() else '없음'} {key:9s} {path.name}")
assert all(p.exists() for p in SRC.values()), '없는 파일이 있다. dataset 폴더를 확인한다.'

# 분석 기간 = 카드 데이터가 제공된 기간
START, END = pd.Timestamp('2025-07-01'), pd.Timestamp('2025-12-31')
ALL_DAYS = pd.date_range(START, END)          # 7/1부터 12/31까지 모든 날짜 (184개)
print(f'\n분석 기간: {START.date()} ~ {END.date()} ({len(ALL_DAYS)}일)')

OK   card      01_대회제공_카드유동인구_202507-202512.zip
OK   weather   02_기상관측_ASOS_AWS_197507-202512.zip
OK   warning   03_기상특보_202507-202512.csv
OK   sales     04_상권매출_행정동_202101-202512.zip
OK   subway    05_지하철승하차_역별_202301-202608.zip
OK   stores    06_상가정보_강남춘천_202510_202512.zip
OK   tour      07_관광방문자_강남춘천_202101-202512.zip
OK   boundary  HangJeongDong_ver20250401.geojson

분석 기간: 2025-07-01 ~ 2025-12-31 (184일)


### 0-3. 여러 번 쓰는 함수

| 함수 | 하는 일 |
|---|---|
| `zip_files` | zip 안의 파일 목록을 한글 이름이 안 깨지게 가져옴 |
| `decode` | 파일의 인코딩(utf-8 / cp949)을 자동으로 알아내서 글자로 바꿈 |
| `log_issue` | 데이터에서 찾은 문제를 기록함 (맨 마지막에 표로 모아서 보여줌) |
| `save` | 표를 parquet 파일로 저장함 |

In [3]:
def zip_name(info):
    """
    zip 안 파일 하나의 이름을 한글이 깨지지 않게 돌려준다.

    zip 파일은 만든 프로그램에 따라 파일 이름을 UTF-8 또는 CP949(윈도우 한글)로 저장한다.
    UTF-8로 저장했으면 zip 안에 '표시(flag)'가 켜져 있어서, 그걸 보고 구분한다.
    """
    if info.flag_bits & 0x800:        # 0x800 = "이 이름은 UTF-8" 표시
        return info.filename
    try:                               # 표시가 없으면 파이썬이 이름을 잘못 읽은 상태 -> CP949로 되돌림
        return info.filename.encode('cp437').decode('cp949')
    except UnicodeError:
        return info.filename


def zip_files(zf):
    """zip 안의 파일들을 {한글 이름: 파일 정보} dictionary로 돌려준다. 폴더는 뺀다."""
    return {zip_name(i): i for i in zf.infolist() if not i.is_dir()}


def decode(raw):
    """
    바이트(raw)를 글자로 바꾼다. utf-8을 먼저 해보고, 안 되면 cp949로 읽는다.

    같은 데이터셋 안에서도 파일마다 인코딩이 다를 수 있어서 필요한다.
    'utf-8-sig'는 파일 맨 앞에 붙은 보이지 않는 표시 문자(BOM)까지 알아서 지워준다.
    돌려주는 값: (글자, 사용한 인코딩)
    """
    for enc in ('utf-8-sig', 'cp949'):
        try:
            return raw.decode(enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError('utf-8도 cp949도 아닌 인코딩이다.')


ISSUES = []   # 찾은 문제를 모아둘 빈 목록

def log_issue(data, problem, action):
    """데이터 문제와 처리 방법을 기록한다.(마지막에 한 눈에 확인 용도)"""
    ISSUES.append({'데이터': data, '문제': problem, '처리': action})
    print(f'[문제 기록] {data}: {problem} → {action}')


SAVED = []    # 저장한 파일 목록

def save(df, name):
    """표를 parquet 파일로 저장하고 크기를 알려준다. 같은 이름으로 다시 저장하면 덮어쓴다."""
    path = OUT / f'{name}.parquet'
    df.to_parquet(path, index=False)     # index=False: 왼쪽의 행 번호(0,1,2...)는 저장하지 않음
    mb = path.stat().st_size / 1e6
    SAVED[:] = [s for s in SAVED if s['파일'] != path.name]    # 예전 기록이 있으면 지우고 새로 기록
    SAVED.append({'파일': path.name, '행': len(df), '열': df.shape[1], '크기(MB)': round(mb, 1)})
    print(f'[저장] {path.name}  {len(df):,}행 × {df.shape[1]}열, {mb:.1f}MB')

## 1. 카드 데이터1 (시간대별)
신한카드 결제 데이터이다. 한 행 = **날짜 × 시간대 × 지역 × 업종 × 성별 × 연령** 조합 하나의 결제 합계이다.
정의서에 따르면 **오프라인 결제만** 들어 있다.

열 이름은 코드에서 쓰기 편하도록 짧은 영어로 바꾼다.

| 원래 열 | 새 이름 | 뜻 |
|---|---|---|
| TA_YMD | `date` | 결제 날짜 |
| TIME_GB | `time_gb` | 시간대: `00_05` / `06_11` / `12_17` / `18_23` (6시간 단위) |
| MCT_SGG_CD | `region` | 가맹점 지역: `강남` / `춘천` |
| MCT_RY_CD | `industry` | 가맹점 업종 (91개) |
| SEX_CCD | `sex` | 성별: 남성 / 여성 / 법인 |
| AGE_CCD | `age` | 연령대: 10대 ~ 90대 / 법인 |
| TS_AT | `amount` | 결제 금액 합계 (원) |
| USE_CNT | `cnt` | 결제 건수 합계 |

### 1-1. 읽기
zip 파일을 풀지 않고 안에 든 `데이터1.txt`를 바로 읽는다. 먼저 **모든 열을 글자로** 읽은 뒤, 1-2에서 숫자와 날짜로 직접 바꾼다.

In [4]:
# zip을 풀지 않고 안에 든 txt 파일을 바로 읽음
with zipfile.ZipFile(SRC['card']) as zf:       # with 블록이 끝나면 zip 파일이 자동으로 닫힘
    files = zip_files(zf)
    name = next(n for n in files if n.endswith('데이터1.txt'))   # 이름이 '데이터1.txt'로 끝나는 파일 찾기
    card1_raw = pd.read_csv(
        zf.open(files[name]),
        sep='\t',           # 열이 탭(Tab)으로 구분된 파일
        encoding='cp949',   # 윈도우 한글 인코딩
        dtype=str,          # 일단 모두 글자로 읽음 -> 숫자, 날짜 변환은 아래에서 직접 (변환 실패를 잡아내려고)
    )

print('크기 (행, 열):', card1_raw.shape)
card1_raw.head()    # 앞 5행 미리보기. 셀의 마지막 줄에 표를 두면 예쁘게 출력됨

크기 (행, 열): (1044710, 8)


,TA_YMD,TIME_GB,MCT_SGG_CD,MCT_RY_CD,SEX_CCD,AGE_CCD,TS_AT,USE_CNT
0,20250704,12_17,서울 강남구,가구,법인,법인,15477163,16
1,20250721,12_17,서울 강남구,가구,법인,법인,308451,38
2,20250703,12_17,서울 강남구,가구,여성,20 대,78970,5
3,20250703,18_23,서울 강남구,가구,남성,20 대,271220,5
4,20250706,18_23,서울 강남구,가구,여성,20 대,260872,5


### 1-2. 정리
열 이름을 짧은 영어로 바꾸고, 날짜와 금액, 건수를 알맞은 형식으로 바꾼다. 지역 이름은 `강남`, `춘천`으로 통일한다.
데이터2도 같은 방식으로 정리하므로 함수(`clean_card`) 하나로 만들어 둔다.

In [5]:
# 데이터1과 데이터2는 열 구성이 거의 같아서, 정리 과정을 함수 하나로 만들어 공통 적용

CARD_COLS = {                     # {원래 이름: 새 이름}
    'TA_YMD': 'date', 'TIME_GB': 'time_gb', 'MCT_SGG_CD': 'region', 'MCT_RY_CD': 'industry',
    'CLN_SGG_CD': 'residence',    # 데이터2에만 있는 열: 고객 거주지(시도)
    'SEX_CCD': 'sex', 'AGE_CCD': 'age', 'TS_AT': 'amount', 'USE_CNT': 'cnt',
}
# 모든 노트북에서 지역 이름을 '강남', '춘천' 두 가지로 통일
REGION_MAP = {'서울 강남구': '강남', '강원 춘천시': '춘천'}


def clean_card(raw):
    df = raw.rename(columns=CARD_COLS)    # 열 이름 바꾸기 (표에 없는 열 이름은 그냥 무시됨)

    # 날짜: '20250701' 같은 글자 -> 날짜형. format을 알려주면 빠르고 정확함
    df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')

    # 금액과 건수: 글자 -> 정수 
    # 숫자가 아닌 값이 섞여 있으면 여기서 에러가 나므로 자동으로 점검됨
    df['amount'] = pd.to_numeric(df['amount']).astype('int64')
    df['cnt'] = pd.to_numeric(df['cnt']).astype('int64')

    df['region'] = df['region'].map(REGION_MAP)                # '서울 강남구' -> '강남'
    df['age'] = df['age'].str.replace(' ', '', regex=False)    # '20 대' -> '20대' (중간 공백 제거)
    return df


card1 = clean_card(card1_raw)
card1.head()

,date,time_gb,region,industry,sex,age,amount,cnt
0,2025-07-04,12_17,강남,가구,법인,법인,15477163,16
1,2025-07-21,12_17,강남,가구,법인,법인,308451,38
2,2025-07-03,12_17,강남,가구,여성,20대,78970,5
3,2025-07-03,18_23,강남,가구,남성,20대,271220,5
4,2025-07-06,18_23,강남,가구,여성,20대,260872,5


### 1-3. 기본 점검
1. 빈 값이 없는지
2. 184일이 모두 있는지
3. 같은 칸(키)이 두 번 나오지 않는지
4. 각 열에 예상한 값만 들어 있는지

In [6]:
print(f'행 수: {len(card1):,}')

# (1) 빈 값이 있는가? -> 모두 0이어야 정상. REGION_MAP에 없는 지역이 있었다면 region이 비어서 여기서 잡힘
print('\n열별 빈 값 개수:')
display(card1.isna().sum().rename('빈 값 개수').to_frame())   # to_frame(): 열 하나짜리(Series)를 표로 바꿈

# (2) 184일이 빠짐없이 있는가?
missing_days = ALL_DAYS.difference(card1['date'].unique())   # 전체 날짜 중 데이터에 없는 날짜
print(f"\n날짜 수: {card1['date'].nunique()} / 빠진 날짜: {list(missing_days)}")

# (3) 같은 칸이 두 번 나오지 않는가?
KEY1 = ['date', 'time_gb', 'region', 'industry', 'sex', 'age']   # 한 행을 구분하는 열 조합 (= 키)
print('키 중복 행:', card1.duplicated(KEY1).sum()) # 중복된 행의 개수를 셈

# (4) 각 열에 어떤 값이 있는가?
for col in ['time_gb', 'region', 'sex', 'age']:
    print(f'{col}: {sorted(card1[col].unique())}')
print('업종 수:', card1['industry'].nunique())

행 수: 1,044,710

열별 빈 값 개수:


,빈 값 개수
date,0
time_gb,0
region,0
industry,0
sex,0
age,0
amount,0
cnt,0



날짜 수: 184 / 빠진 날짜: []


키 중복 행: 0


time_gb: ['00_05', '06_11', '12_17', '18_23']
region: ['강남', '춘천']
sex: ['남성', '법인', '여성']
age: ['10대', '20대', '30대', '40대', '50대', '60대', '70대', '80대', '90대', '법인']
업종 수: 91


### 1-4. 5건 미만 마스킹
결제 건수(`cnt`)의 최솟값이 5라면, 결제가 5건 미만인 행은 삭제됐다고 생각할 수 있음

왜 중요하냐면: 폭염 때문에 어떤 칸의 결제가 8건에서 3건으로 줄면, 그 칸은 행 자체가 사라진다.<br>
그러면 "폭염일 평균 매출"을 계산할 때 줄어든 3건이 빠지므로 감소폭이 실제보다 작게 나온다.<br>
삭제된 행을 생각하지 않고 전체로 평균을 내면 삭제된 애들이 평균치를 많이 내리는데 집계되지 않아서 실제로 매출 타격이 큰 애들이 그렇지 않아보일 수 있다.<br>
그래서 3장에서 "184일 내내 빠짐없이 나오는 칸"에 표시(`is_balanced`)를 붙여서 따로 볼 수 있게 한다.

In [7]:
print('건수 최솟값:', card1['cnt'].min())
print('금액 최솟값:', card1['amount'].min())

# 날짜마다 남아 있는 행 수
rows_per_day = card1.groupby('date').size()
print('\n일별 행 수 요약 (count=날짜 수, mean=평균, min=최소, max=최대):')
display(rows_per_day.describe().round(0).rename('일별 행 수').to_frame())
print('\n행이 가장 적은 5일:')
display(rows_per_day.nsmallest(5).rename('행 수').to_frame())

log_issue('카드1',
          f'결제 5건 미만인 칸은 삭제돼 있음. 일별 행 수가 {rows_per_day.min():,}~{rows_per_day.max():,}개로 변동',
          '184일 내내 나오는 칸에 is_balanced=True 표시 (3장)')

건수 최솟값: 5
금액 최솟값: 6

일별 행 수 요약 (count=날짜 수, mean=평균, min=최소, max=최대):


,일별 행 수
count,184.0
mean,5678.0
std,257.0
min,4268.0
25%,5656.0
50%,5749.0
75%,5833.0
max,6023.0



행이 가장 적은 5일:


,행 수
date,
2025-10-06,4268
2025-10-07,4850
2025-10-05,5058
2025-12-14,5080
2025-12-07,5096


[문제 기록] 카드1: 결제 5건 미만인 칸은 삭제돼 있음. 일별 행 수가 4,268~6,023개로 변동 → 184일 내내 나오는 칸에 is_balanced=True 표시 (3장)


### 1-5. 법인 결제와 초대형 결제
`법인`은 성별과 연령 **두 열에 동시에** 들어가야 함.

In [8]:
is_corp_sex = card1['sex'].eq('법인')     # .eq('법인') = "값이 '법인'인가?" -> True/False 목록
is_corp_age = card1['age'].eq('법인')
# & 는 '그리고', ~ 는 '아닌 것'
print('성별만 법인:', (is_corp_sex & ~is_corp_age).sum(), '| 연령만 법인:', (~is_corp_sex & is_corp_age).sum())

corp_share = card1.loc[is_corp_sex, 'amount'].sum() / card1['amount'].sum()
print(f'법인 결제가 전체 금액에서 차지하는 비중: {corp_share:.1%}')
log_issue('카드1', f'법인 결제가 전체 금액의 {corp_share:.1%}',
          "날씨 분석에서는 기본적으로 제외 (sex != '법인'). 02번에서 따로 확인")

# 금액이 가장 큰 칸 10개는 어떤 업종일까?
card1.nlargest(10, 'amount')[KEY1 + ['amount', 'cnt']]

# 세금공과금, zz_나머지는 빼는데, zz_나머지는 특이 업종이라서 기타로 묶인 거 같음. 이런 애들은 분해해볼 수 없나??

성별만 법인: 0 | 연령만 법인: 0
법인 결제가 전체 금액에서 차지하는 비중: 15.9%
[문제 기록] 카드1: 법인 결제가 전체 금액의 15.9% → 날씨 분석에서는 기본적으로 제외 (sex != '법인'). 02번에서 따로 확인


,date,time_gb,region,industry,sex,age,amount,cnt
266988,2025-08-06,12_17,강남,ZZ_나머지,법인,법인,520512895491,16270
475447,2025-09-04,12_17,강남,ZZ_나머지,법인,법인,299091722044,19579
840662,2025-12-29,12_17,강남,세금공과금,남성,60대,289646721166,7773
671046,2025-10-10,12_17,강남,ZZ_나머지,법인,법인,173658599830,13753
95643,2025-07-04,12_17,강남,ZZ_나머지,법인,법인,160960081300,16069
983605,2025-12-04,12_17,강남,ZZ_나머지,법인,법인,118230904057,20916
741221,2025-11-06,12_17,강남,ZZ_나머지,법인,법인,95442109367,16879
953360,2025-12-01,06_11,강남,세금공과금,법인,법인,64874808897,261
541960,2025-10-27,06_11,강남,세금공과금,법인,법인,57981146108,247
742126,2025-11-10,12_17,강남,세금공과금,법인,법인,43090355024,646


금액 상위 10개 칸은 모두 `ZZ_나머지`(분류 불명)와 `세금공과금` 이다. 한 칸에 하루 5,205억 원이 찍힐 만큼 큰 결제라서, 날씨 분석에 섞이면 결과가 이 업종들에 끌려간다.
이런 업종을 분석 대상에서 빼야한다.

### 1-6. ZZ_나머지는 어떤 결제인가
`ZZ_나머지`는 신한카드 업종 분류 90개 어디에도 들어가지 않는 가맹점을 모아 둔 칸이다. 데이터 정의서에도 설명이 없고, 가맹점 이름이나 주소가 없어서 안에 어떤 업장이 있는지는 알 수 없다.
대신 다른 업종과 비교해서 어떤 성격의 결제인지만 확인한다.
1. 전체 금액과 건수 중 몇 %인가 (지역별)
2. 1건당 금액과 법인 결제 비중
3. 하루 중 언제 결제되나 (시간대)

In [9]:
# ZZ_나머지(분류 불명)가 어떤 결제인지, 다른 업종 전체와 같은 기준으로 비교한다.
is_zz = card1['industry'].eq('ZZ_나머지')     # ZZ_나머지 행이면 True


# (1) 전체 금액과 건수 중 ZZ_나머지 비중 (지역별 + 전체)
def zz_share(df):
    """df 안에서 ZZ_나머지가 차지하는 금액 비중과 건수 비중을 돌려준다."""
    zz = df['industry'].eq('ZZ_나머지')
    return pd.Series({
        '금액 비중': df.loc[zz, 'amount'].sum() / df['amount'].sum(),
        '건수 비중': df.loc[zz, 'cnt'].sum() / df['cnt'].sum(),
    })

share = card1.groupby('region')[['industry', 'amount', 'cnt']].apply(zz_share)   # 지역별로 한 줄씩 (필요한 열만 골라서 넘김)
share.loc['전체'] = zz_share(card1)                  # 맨 아래에 전체 한 줄 추가
print('ZZ_나머지 비중:')
display(share.map('{:.1%}'.format))                  # map: 표의 모든 값을 '26.1%' 같은 글자형식으로 바꿔서 보기 좋게


# (2) 1건당 금액과 법인 결제 비중: ZZ_나머지 vs 다른 업종 전체
def profile(df):
    """금액, 건수, 1건당 금액, 법인 결제 비중을 한 번에 계산한다."""
    return pd.Series({
        '금액(억 원)':    df['amount'].sum() / 1e8,
        '건수(만 건)':    df['cnt'].sum() / 1e4,
        '1건당 금액(원)': df['amount'].sum() / df['cnt'].sum(),       # 건당 평균이 아니라 "전체 금액 / 전체 건수"
        '법인 결제 비중': df.loc[df['sex'].eq('법인'), 'amount'].sum() / df['amount'].sum(),
    })

compare = pd.DataFrame({'ZZ_나머지': profile(card1[is_zz]), '다른 업종 전체': profile(card1[~is_zz])})  
print('\nZZ_나머지 vs 다른 업종:')
display(compare.round(2))


# (3) 시간대별 금액 비중: 하루 중 언제 결제되나
def time_mix(df):
    """시간대별 금액 비중(합이 1)을 돌려준다."""
    s = df.groupby('time_gb')['amount'].sum()
    return s / s.sum()

time_compare = pd.DataFrame({'ZZ_나머지': time_mix(card1[is_zz]), '다른 업종 전체': time_mix(card1[~is_zz])})
print('\n시간대별 금액 비중:')
display(time_compare.map('{:.1%}'.format))

ZZ_나머지 비중:


,금액 비중,건수 비중
region,,
강남,26.3%,18.3%
춘천,24.4%,3.2%
전체,26.1%,17.0%



ZZ_나머지 vs 다른 업종:


,ZZ_나머지,다른 업종 전체
금액(억 원),93732.38,264926.19
건수(만 건),10422.20,50812.84
1건당 금액(원),89935.32,52137.65
법인 결제 비중,0.31,0.11



시간대별 금액 비중:


,ZZ_나머지,다른 업종 전체
time_gb,,
00_05,4.3%,2.7%
06_11,26.6%,26.6%
12_17,55.3%,53.6%
18_23,13.7%,17.1%


**ZZ_나머지는 카드 데이터1 금액의 약 26%** 를 차지한다. 업종 하나가 4분의 1이 넘는다.
- 1건당 금액이 약 9만 원으로, 다른 업종(약 5.2만 원)보다 1.7배 크다.
- 법인 결제 비중이 약 31%로, 다른 업종(약 11%)의 3배 가까이 된다.
- 춘천은 **건수로는 3%뿐인데 금액으로는 24%** 이다. 소수의 아주 큰 결제가 들어 있다.
- 시간대 구성은 다른 업종과 비슷하지만, 저녁(18~23시) 비중이 조금 낮다.

그래서 일반 소비보다는 사업체 거래나 분류가 애매한 서비스가 섞인 것으로 짐작된다. 무엇인지 모르는데 금액이 커서 분석에 영향을 준다고 생각한다.그래서 분석 대상에서 뺀다.
(`is_target=False`, 이유: 분류 불명)

## 2. 카드 데이터2 (고객 거주지별)
데이터1과 비슷하지만 시간대 대신 고객 거주지(시도) 가 있고, 법인 결제는 빠져 있다.

| 새 열 | 뜻 |
|---|---|
| `residence` | 고객 거주지 시도 (서울, 경기, 강원 등 16개 + `정보없음`) |

### 2-1. 읽기와 정리
1-2에서 만든 `clean_card` 함수로 똑같이 정리하고, 1-3과 같은 점검을 한 번에 한다.

In [10]:
with zipfile.ZipFile(SRC['card']) as zf:
    files = zip_files(zf)
    name = next(n for n in files if n.endswith('데이터2_수정.txt'))
    card2_raw = pd.read_csv(zf.open(files[name]), sep='\t', encoding='cp949', dtype=str)

card2 = clean_card(card2_raw)    # 1-2에서 만든 함수를 그대로 다시 씀

KEY2 = ['date', 'region', 'industry', 'residence', 'sex', 'age']
print(f'행 수: {len(card2):,}')
print('빈 값 합계:', card2.isna().sum().sum())
print('날짜 수:', card2['date'].nunique(), '| 키 중복 행:', card2.duplicated(KEY2).sum())
print('성별:', sorted(card2['sex'].unique()), '| 건수 최솟값:', card2['cnt'].min())
print('거주지:', sorted(card2['residence'].unique()))
card2.head()

행 수: 1,806,800
빈 값 합계: 0


날짜 수: 184 | 키 중복 행: 0
성별: ['남성', '여성'] | 건수 최솟값: 5
거주지: ['강원', '경기', '경남', '경북', '대구', '대전', '부산', '서울', '세종', '울산', '인천', '전남광주', '전북', '정보없음', '제주', '충남', '충북']


,date,region,industry,residence,sex,age,amount,cnt
0,2025-07-14,춘천,가구,강원,남성,50대,77544,18
1,2025-07-23,강남,가구,서울,여성,30대,651570,13
2,2025-07-11,춘천,가전,강원,여성,30대,4239967,16
3,2025-07-19,춘천,가전,강원,남성,30대,828641,18
4,2025-07-23,춘천,가전,강원,여성,30대,443864,11


### 2-2. 데이터1과 같은 결제를 다르게 자른 것인가?
데이터1에서 법인을 뺀 **날짜별 총액**과 데이터2의 **날짜별 총액**을 비교한다.
둘이 같다면 두 데이터는 "같은 결제를 다른 기준으로 나눈 것"이다.

In [11]:
d1 = card1.loc[card1['sex'] != '법인'].groupby('date')['amount'].sum()   # 데이터1: 법인 빼고 날짜별 합계
d2 = card2.groupby('date')['amount'].sum()                                # 데이터2: 날짜별 합계

compare = pd.DataFrame({'데이터1_법인제외': d1, '데이터2': d2})
compare['차이(원)'] = compare['데이터2'] - compare['데이터1_법인제외']
compare['비율'] = compare['데이터2'] / compare['데이터1_법인제외']

print('비율 최솟값:', compare['비율'].min())
print('비율 최댓값:', compare['비율'].max())
print(f"하루 최대 차이: {compare['차이(원)'].abs().max():,}원")

# 비율이 184일 모두 1에 아주 가까운지 확인 (0.0001% 이내). 다르면 여기서 에러가 남
assert (compare['비율'] - 1).abs().max() < 1e-6, '두 데이터의 일별 총액이 다르다!'
print('→ 184일 모두 일치: 같은 결제 데이터이다.')
compare.head()

비율 최솟값: 0.9999999989851817
비율 최댓값: 1.0000000009167869
하루 최대 차이: 115원
→ 184일 모두 일치: 같은 결제 데이터이다.


,데이터1_법인제외,데이터2,차이(원),비율
date,,,,
2025-07-01,172478422558,172478422584,26,1.0
2025-07-02,138581440228,138581440278,50,1.0
2025-07-03,140432373944,140432374024,80,1.0
2025-07-04,162869675142,162869675170,28,1.0
2025-07-05,97759769794,97759769799,5,1.0


## 3. 분석용 표시 열 추가
카드 데이터 두 개에 **분석할 때 자주 거르는 기준**을 열로 붙여 둔다. 그러면 뒤 노트북에서는 `is_target`, `is_balanced` 같은 열로 바로 거를 수 있다.

### 3-1. 업종 묶음과 분석 대상 여부
업종이 91개나 되면 쪼개서 볼 때 칸이 너무 잘게 나뉜다. 비슷한 업종끼리 **14개 묶음**(`industry_group`)으로 묶는다.

또 **날씨와 상관없는 결제**는 분석에서 뺀다(`is_target=False`). 예를 들어 `세금공과금`은 금액이 매우 크지만 날씨 때문에 세금을 덜 내지는 않는다.
이런 업종이 섞여 있으면 전체 매출 추이가 이 업종들의 움직임에 영향을 받는다.

In [12]:
# {묶음 이름: [업종 목록]} — 날씨 분석 대상 업종
TARGET_GROUPS = {
    '음식점':          ['한식', '중식', '일식', '양식', '기타요식', '패스트푸드', '제과점'],
    '카페':            ['커피전문점'],
    '주점/유흥':       ['유흥업소'],
    '편의점':          ['편의점'],
    '식료품':          ['할인점/슈퍼마켓/양판점', '농수산물', '정육점', '기타식품', '주류판매'],
    '대형유통':        ['백화점', '면세점', '쇼핑몰', '기타유통', '전용매장'],
    '패션/뷰티':       ['의복/의류', '패션잡화', '화장품', '시계/귀금속'],
    '생활/취미용품':   ['가구', '가전', '생활잡화/수입상품점', '인테리어/건축자재/주방기구', '사무기기/문구용품',
                        '서점', '문화용품', '악기/음반', '완구/아동용자전거', '수제용품점', '화원', '애완동물',
                        '중고품판매점', '스포츠/레저용품'],
    '여가/오락':       ['노래방', '게임방/오락실', '영화/공연', '종합레저타운/놀이동산', '독서실'],
    '스포츠시설':      ['실내/실외골프장', '스포츠시설', '헬스장', '싸우나/목욕탕'],
    '숙박/여행':       ['호텔/콘도', '모텔,여관,기타숙박', '여행사/항공사'],
    '교통/차량':       ['택시', '고속버스/철도/여객선', '주차장', '주유소', '자동차서비스', '자동차용품', '오토바이'],
    '미용/생활서비스': ['미용실', '미용서비스', '안마/마사지', '세탁소'],
    '의료':            ['일반병원', '종합병원', '치과병원', '한의원', '약국', '기타의료', '동물병원'],
}


# {제외 이유: [업종 목록]} — 날씨와 무관하다고 보고 분석에서 빼는 업종
EXCLUDED = {
    '분류 불명':          ['ZZ_나머지', '체인점'],    # 체인점: 무슨 체인점인지 알 수 없음 (아래 "체인점을 제외한 이유")
    '공과금/정기 납부':   ['세금공과금', '보험', '통신요금(이동,시내전화)', '통신요금(PC통신,무선호출)',
                           'LPG가스', '학교등록금', '유치원', '학원/학습지'],
    '고액/비일상 거래':   ['수입자동차', '중고차판매', '부동산중개', '예식장/결혼서비스', '장례식장/묘지/장의사'],
    '중개/비대면성 결제': ['결제대행(PG)', '방문판매/다단계판매', '상품권/복권'],
    '기업/전문 서비스':   ['법률/사무서비스', '회계/변리서비스', '연구/번역서비스', '컴퓨터/소프트웨어'],
    '공공기관':           ['보건소'],
}

# 사전 두 개를 표 하나로 합친다. 한 행 = 업종 하나
rows = [{'industry': ind, 'industry_group': grp, 'is_target': True, 'exclude_reason': ''}
        for grp, inds in TARGET_GROUPS.items() for ind in inds]
rows += [{'industry': ind, 'industry_group': '제외', 'is_target': False, 'exclude_reason': why}
         for why, inds in EXCLUDED.items() for ind in inds]
industry_map = pd.DataFrame(rows)

# 점검: 데이터에 있는 업종이 빠짐없이, 한 번씩만 분류됐는가?
all_inds = set(card1['industry']) | set(card2['industry'])    # | = 합집합
mapped = set(industry_map['industry'])
assert industry_map['industry'].is_unique, '두 번 분류된 업종이 있다.'
assert all_inds == mapped, f'분류 안 된 업종: {all_inds - mapped} / 데이터에 없는 업종: {mapped - all_inds}'
print(f"업종 {len(all_inds)}개 모두 분류 완료 (분석 대상 {industry_map['is_target'].sum()}개, 제외 {(~industry_map['is_target']).sum()}개)")

# 제외 업종이 금액에서 차지하는 비중 -> 얼마나 큰 덩어리를 빼는지 확인
excluded_inds = industry_map.loc[~industry_map['is_target'], 'industry']
excl_share = card1.loc[card1['industry'].isin(excluded_inds), 'amount'].sum() / card1['amount'].sum()
print(f'제외 업종이 데이터1 전체 금액에서 차지하는 비중: {excl_share:.1%}')

industry_map.to_csv(OUT / 'industry_map.csv', index=False, encoding='utf-8-sig')  # utf-8-sig: 엑셀에서 열어도 한글 안 깨짐
industry_map.groupby('industry_group').size().rename('업종 수')

업종 91개 모두 분류 완료 (분석 대상 68개, 제외 23개)
제외 업종이 데이터1 전체 금액에서 차지하는 비중: 62.9%


industry_group
교통/차량        7
대형유통         5
미용/생활서비스     4
생활/취미용품     14
숙박/여행        3
스포츠시설        4
식료품          5
여가/오락        5
음식점          7
의료           7
제외          23
주점/유흥        1
카페           1
패션/뷰티        4
편의점          1
Name: 업종 수, dtype: int64

In [ ]:
# 카드 데이터 두 개에 업종 묶음과 분석 대상 여부 열을 붙인다. (그냥 열 추가라고 생각하기)
# merge = 두 표를 공통 열(industry) 기준으로 옆으로 붙이기
cols = ['industry', 'industry_group', 'is_target']
card1 = card1.merge(industry_map[cols], on='industry', how='left')
card2 = card2.merge(industry_map[cols], on='industry', how='left')
print('붙이기 실패(빈 값):', card1['is_target'].isna().sum(), card2['is_target'].isna().sum())

붙이기 실패(빈 값): 0 0


#### 체인점을 제외한 이유
`체인점`은 무슨 체인점을 말하는지 알 수 없어서(음식점인지, 로드샵 같은 판매점인지, 다른 서비스인지) 일단 분석 대상에서 뺀다(`is_target=False`, 이유: 분류 불명).

- 카드사는 가맹점마다 업종을 하나만 붙인다. 그래서 한식 프랜차이즈는 `한식`, 커피 프랜차이즈는 `커피전문점`으로 이미 들어가 있고, `체인점`에 한 번 더 들어가지 않는다.
- 데이터 정의서에도 `체인점`이 무엇인지 설명이 없고, 가맹점 이름이나 주소가 없어서 안을 들여다볼 수 없다.
- 아래 코드로 결제의 모양(규모, 1건당 금액, 시간대, 주말)만 다른 업종과 비교해 본다.

In [14]:
# 체인점과 성격이 비슷할 만한 업종(음식점, 편의점, 유통)을 같은 기준으로 비교한다.
# 가맹점 이름이 없어서 "무슨 가게인지"는 알 수 없고, 결제의 모양만 비교할 수 있다.
COMPARE = ['체인점', '한식', '패스트푸드', '커피전문점', '편의점', '할인점/슈퍼마켓/양판점', '기타유통', '화장품']

personal = card1[card1['sex'] != '법인']      # 개인 결제만 (법인은 성격이 달라서 뺌)


def chain_profile(df):
    """금액, 1건당 금액, 시간대 비중, 주말/평일 비율을 한 줄로 요약한다."""
    time_share = df.groupby('time_gb')['amount'].sum() / df['amount'].sum()
    dow = df['date'].dt.dayofweek                                  # 0=월 ... 6=일
    daily = df.groupby('date')['amount'].sum()                     # 날짜별 합계
    weekend = daily[daily.index.dayofweek >= 5].mean()             # 주말 하루 평균
    weekday = daily[daily.index.dayofweek < 5].mean()              # 평일 하루 평균
    return pd.Series({
        '금액(억 원)':      df['amount'].sum() / 1e8,
        '강남(억 원)':      df.loc[df['region'].eq('강남'), 'amount'].sum() / 1e8,
        '춘천(억 원)':      df.loc[df['region'].eq('춘천'), 'amount'].sum() / 1e8,
        '1건당 금액(원)':   df['amount'].sum() / df['cnt'].sum(),
        '12~17시 비중':     time_share.get('12_17', 0),
        '18~23시 비중':     time_share.get('18_23', 0),
        '주말/평일':        weekend / weekday,
    })


profile = pd.DataFrame({ind: chain_profile(personal[personal['industry'].eq(ind)]) for ind in COMPARE}).T   # .T = 행과 열 뒤집기
display(profile.round(2))

# 체인점이 며칠에 나오는지: 5건 미만 마스킹 때문에 작은 업종은 빠지는 날이 많음
chain_days = card1.loc[card1['industry'].eq('체인점'), 'date'].nunique()
print(f'\n체인점이 데이터에 나오는 날: {chain_days}일 / {len(ALL_DAYS)}일')

,금액(억 원),강남(억 원),춘천(억 원),1건당 금액(원),12~17시 비중,18~23시 비중,주말/평일
체인점,4.21,3.84,0.36,9730.05,0.44,0.33,0.81
한식,8889.73,6621.44,2268.28,34300.39,0.39,0.48,1.03
패스트푸드,499.93,397.15,102.78,11397.13,0.47,0.34,1.07
커피전문점,3750.53,3591.37,159.16,8477.54,0.54,0.18,1.11
편의점,3094.95,2266.82,828.14,6337.43,0.32,0.35,0.82
할인점/슈퍼마켓/양판점,13350.62,10904.76,2445.86,12657.11,0.55,0.32,1.31
기타유통,283.25,206.68,76.57,51002.61,0.59,0.09,0.65
화장품,787.05,753.96,33.09,62250.44,0.58,0.28,0.57



체인점이 데이터에 나오는 날: 184일 / 184일


`체인점`은 6개월 동안 **약 4억 원**(강남 약 3.8억, 춘천 약 0.4억)뿐이다. `한식`(약 8,890억 원)의 2천분의 1도 안 된다. 한식이나 커피 체인이 여기에 들어 있다면 이렇게 작을 수 없다.
- 1건당 약 9,700원, 주말/평일 약 0.81배로 모양은 편의점이나 패스트푸드와 비슷하지만, 그것만으로 무슨 가게인지 정할 수는 없다.
- 무엇인지 모르면 "날씨 때문에 줄었다"고 해석할 수 없어서 `ZZ_나머지`와 같은 이유(분류 불명)로 뺀다. 금액이 작아서 빼도 다른 결과는 거의 바뀌지 않는다.

### 3-2. 균형 패널 표시 (`is_balanced`)
**184일 내내 한 번도 빠지지 않은 칸**에만 `True`를 붙인다. (칸 = 날짜를 뺀 나머지 조합, 예: "강남, 한식, 12_17, 여성, 30대")
- 더운 날 결제가 8건에서 3건으로 줄면 그 칸은 그날 데이터에서 사라진다. 가장 많이 줄어든 칸이 빠진 채로 평균을 내니 감소가 작게 보인다.
- is_balanced=True인 칸은 매일 결제가 많아서 사라지지 않는다. 대신 골프장처럼 작은 칸은 처음부터 빠져서, 날씨에 민감한 곳을 놓칠 수 있다.

In [15]:
def add_balanced(df, cell_key):
    # transform('nunique'): 칸마다 '등장한 날짜 수'를 세어서, 원래 표와 같은 행 수로 돌려줌
    n_days = df.groupby(cell_key)['date'].transform('nunique')
    df['is_balanced'] = n_days.eq(len(ALL_DAYS))   # 184일이면 True
    return df

card1 = add_balanced(card1, ['time_gb', 'region', 'industry', 'sex', 'age'])
card2 = add_balanced(card2, ['region', 'industry', 'residence', 'sex', 'age'])

# 분석 대상(개인 결제 + 분석 대상 업종) 중에서 균형 패널이 차지하는 비중
for name, df in [('카드1', card1), ('카드2', card2)]:
    base = df[df['is_target'] & (df['sex'] != '법인')]
    row_share = base['is_balanced'].mean()                                        # True=1, False=0이라 평균 = 비율
    amt_share = base.loc[base['is_balanced'], 'amount'].sum() / base['amount'].sum()
    print(f'{name}: 균형 패널 = 행의 {row_share:.1%}, 금액의 {amt_share:.1%}')

카드1: 균형 패널 = 행의 40.9%, 금액의 81.8%
카드2: 균형 패널 = 행의 28.4%, 금액의 76.1%


### 3-3. 저장

In [16]:
save(card1, 'card1')
save(card2, 'card2')

[저장] card1.parquet  1,044,710행 × 11열, 9.8MB


[저장] card2.parquet  1,806,800행 × 11열, 16.4MB


## 4. SK 유동인구
50m × 50m 격자마다 **한 달 동안의 하루 평균 유동인구**를 담은 데이터이다. (값 = 월 일평균, 정의서 확인)
세 종류가 있고 각각 6개월치(월별 파일)이다.

| 종류 | 값 열 | 내용 |
|---|---|---|
| `age` | `MAN_FLOW_POP_CNT_10G` ~ `WMAN_FLOW_POP_CNT_60GU` | 남녀 × 10대~60대 이상 (12개) |
| `time` | `TMST_00` ~ `TMST_23` | 0시~23시 (24개) |
| `wkdy` | `FLOW_POP_CNT_MON` ~ `_SUN` | 월~일 (7개) |

공통 열: `ym`(기준 연월), `block_cd`(집계구 코드), `x`, `y`(격자 좌표, UTM-K = EPSG:5179)

### 4-1. 읽기
`age`, `time`, `wkdy` 세 종류를 6개월치씩, 모두 18개 파일을 읽어서 종류별로 이어 붙인다.
12월 `time`, `wkdy` 파일은 **모든 행이 두 번씩** 들어 있어서, 완전히 같은 행은 하나만 남긴다.

In [17]:
FLOW_KINDS = ['age', 'time', 'wkdy']
MONTHS = ['202507', '202508', '202509', '202510', '202511', '202512']

flow = {}   # 결과를 {'age': 표, 'time': 표, 'wkdy': 표} 형태로 담을 사전
with zipfile.ZipFile(SRC['card']) as zf:
    files = zip_files(zf)
    for kind in FLOW_KINDS:
        parts = []                      # 월별 표를 모아둘 목록
        for ym in MONTHS:
            name = next(n for n in files if n.endswith(f'flow_{kind}_pop_{ym}.csv'))
            df = pd.read_csv(zf.open(files[name]), sep='|', encoding='utf-8-sig',
                             dtype={'STD_YM': str, 'BLOCK_CD': str})   # 코드는 글자로 읽음 (숫자로 읽으면 앞자리 0이 사라질 수 있음)

            n_dup = df.duplicated().sum()     # 모든 열 값이 완전히 같은 행의 수
            if n_dup:
                log_issue(f'유동_{kind}', f'{ym} 파일에 완전히 같은 행이 {n_dup:,}개 (모든 행이 두 번씩 들어 있음)', '중복 제거')
                df = df.drop_duplicates()
            parts.append(df)
        flow[kind] = pd.concat(parts, ignore_index=True)    # 6개월치를 위아래로 이어 붙이기
        print(f'{kind}: {flow[kind].shape}')

age: (596252, 16)


[문제 기록] 유동_time: 202512 파일에 완전히 같은 행이 92,545개 (모든 행이 두 번씩 들어 있음) → 중복 제거
time: (570805, 28)


[문제 기록] 유동_wkdy: 202512 파일에 완전히 같은 행이 121,267개 (모든 행이 두 번씩 들어 있음) → 중복 제거
wkdy: (720806, 11)


### 4-2. 강남, 춘천 격자만 남기기
`BLOCK_CD` 앞 5자리는 시군구 코드이다. `11230`=강남구, `32010`=춘천시이고, 그 밖의 코드는 경계에 걸친 격자가 이웃 시군구로 들어간 것이라 뺀다.
남긴 격자에는 지역(`region`)과 행정동 코드(`adm_cd`, 앞 8자리) 열을 붙인다.

In [18]:
FLOW_REGION = {'11230': '강남', '32010': '춘천'}
n_other = {}

for kind in FLOW_KINDS:
    df = flow[kind]
    sgg = df['BLOCK_CD'].str[:5]                  # .str[:5] = 글자의 앞 5자리
    other = ~sgg.isin(list(FLOW_REGION))          # 강남도 춘천도 아닌 행
    n_other[kind] = other.sum()
    print(f'{kind}: 강남, 춘천 외 코드 {other.sum()}행 제외 → {sorted(sgg[other].unique())}')

    df = df.loc[~other].copy()                    # .copy(): 잘라낸 표를 독립된 새 표로 만듦 (원본과 엮이지 않게)
    df = df.rename(columns={'STD_YM': 'ym', 'BLOCK_CD': 'block_cd', 'X_COORD': 'x', 'Y_COORD': 'y'})
    df['region'] = df['block_cd'].str[:5].map(FLOW_REGION)
    df['adm_cd'] = df['block_cd'].str[:8]         # 앞 8자리 = 행정동 코드 (통계청)
    value_cols = [c for c in df.columns if c not in ('ym', 'block_cd', 'x', 'y', 'region', 'adm_cd')]
    flow[kind] = df[['region', 'ym', 'block_cd', 'adm_cd', 'x', 'y'] + value_cols]   # 열 순서 정리

log_issue('유동', f'경계에 걸친 격자가 이웃 시군구 코드로 들어 있음 {n_other}', '강남과 춘천 코드만 남김')

age: 강남, 춘천 외 코드 337행 제외 → ['11030', '11040', '11220', '11240', '31370', '32310', '32370', '32380']


time: 강남, 춘천 외 코드 330행 제외 → ['11030', '11040', '11220', '11240', '31370', '32310', '32370', '32380']


wkdy: 강남, 춘천 외 코드 641행 제외 → ['11030', '11040', '11220', '11240', '31370', '32310', '32370', '32380']


[문제 기록] 유동: 경계에 걸친 격자가 이웃 시군구 코드로 들어 있음 {'age': 337, 'time': 330, 'wkdy': 641} → 강남과 춘천 코드만 남김


### 4-3. 기본 점검
- 같은 달, 같은 격자가 두 번 나오지 않는지(키 중복), 음수 값이나 빈 값이 없는지 확인한다. 모두 0이어야 정상이다.
- 달마다 격자 수가 비슷한지도 본다. 한 달만 크게 줄었다면 그 달 파일에 문제가 있다는 뜻이다.

In [19]:
for kind, df in flow.items():
    values = df.drop(columns=['region', 'ym', 'block_cd', 'adm_cd', 'x', 'y'])
    print(f"== {kind}: 키 중복 {df.duplicated(['ym', 'block_cd', 'x', 'y']).sum()}행, "
          f"음수 값 {(values < 0).sum().sum()}개, 빈 값 {values.isna().sum().sum()}개")
    # 월 × 지역별 격자 수. 달마다 비슷해야 정상
    display(df.pivot_table(index='ym', columns='region', values='block_cd', aggfunc='count'))

== age: 키 중복 0행, 음수 값 0개, 빈 값 0개


region,강남,춘천
ym,,
202507,15250,82819
202508,15299,83757
202509,15350,84060
202510,15383,85309
202511,15414,84976
202512,15412,82886


== time: 키 중복 0행, 음수 값 0개, 빈 값 0개


region,강남,춘천
ym,,
202507,15251,78791
202508,15301,80015
202509,15349,79678
202510,15387,82436
202511,15418,80351
202512,15421,77077


== wkdy: 키 중복 0행, 음수 값 0개, 빈 값 0개


region,강남,춘천
ym,,
202507,15369,102349
202508,15413,103436
202509,15449,104306
202510,15486,105046
202511,15510,106630
202512,15523,105648


### 4-4. 주말이 정확히 0인 격자
평일에는 사람이 있는데 토요일과 일요일만 **정확히 0**인 격자를 센다.
- 사무실만 있는 곳이라 주말에 정말 아무도 없을 수도 있고,
- 사람이 적어서 카드 데이터처럼 마스킹 된 것일 수도 있다.

In [20]:
# 요일 데이터에서 '평일엔 사람이 있는데 토요일과 일요일은 정확히 0'인 격자가 얼마나 되는지
w = flow['wkdy']
weekday_sum = w[['FLOW_POP_CNT_MON', 'FLOW_POP_CNT_TUS', 'FLOW_POP_CNT_WED',
                 'FLOW_POP_CNT_THU', 'FLOW_POP_CNT_FRI']].sum(axis=1)       # axis=1: 가로 방향(한 행 안에서) 합계
weekend_zero = w['FLOW_POP_CNT_SAT'].eq(0) & w['FLOW_POP_CNT_SUN'].eq(0) & weekday_sum.gt(0)
print(f'평일 > 0 인데 주말 = 0 인 격자: {weekend_zero.sum():,}개 ({weekend_zero.mean():.1%})')
display(w.loc[weekend_zero].groupby('region').size().rename('격자 수').to_frame())

log_issue('유동_wkdy', f'평일엔 유동이 있는데 토요일과 일요일이 정확히 0인 격자가 {weekend_zero.mean():.1%}',
          '실제 0(업무지구)인지 마스킹인지 03번 노트북에서 확인')
log_issue('유동', 'age/time/wkdy 파일마다 격자 수가 다름 (같은 달이라도)', '세 표를 격자 단위로 합칠 때 주의. 03번에서 확인')

평일 > 0 인데 주말 = 0 인 격자: 41,670개 (5.8%)


,격자 수
region,
강남,202
춘천,41468


[문제 기록] 유동_wkdy: 평일엔 유동이 있는데 토요일과 일요일이 정확히 0인 격자가 5.8% → 실제 0(업무지구)인지 마스킹인지 03번 노트북에서 확인
[문제 기록] 유동: age/time/wkdy 파일마다 격자 수가 다름 (같은 달이라도) → 세 표를 격자 단위로 합칠 때 주의. 03번에서 확인


### 4-5. 저장

In [21]:
for kind in FLOW_KINDS:
    save(flow[kind], f'flow_{kind}')

[저장] flow_age.parquet  595,915행 × 18열, 11.0MB


[저장] flow_time.parquet  570,475행 × 30열, 16.6MB


[저장] flow_wkdy.parquet  720,165행 × 13열, 11.0MB


## 5. 기상 관측
기상 zip 안의 **`통합자료/` 폴더**를 쓴다. 연월별로 쪼개진 파일(`원자료/`, `시간자료/`)을 지점별로 이미 합쳐둔 것이고, 모든 지점이 **매년 7~12월만** 있다.

| 지점 | 종류 | 지역 | 기간 | 시간자료 | 비고 |
|---|---|---|---|---|---|
| 서울 108 | ASOS | 강남 | 1975~2025 | O | 서울 유일 ASOS (종로구). 평년값 계산용 |
| 강남 400 | AWS | 강남 | 2015~2025 | O | 강남구 일원동. 강남 대표 후보 |
| 서초 401, 송파 403 | AWS | 강남 | 2015~2025 | O | 인접구 비교용 |
| 춘천 101 | ASOS | 춘천 | 1975~2025 | O | 춘천 대표. 평년값 계산용 |
| 북춘천 93 | ASOS | 춘천 | 2016~2025 | X | 2016년 10월 신설 |
| 남이섬 675, 남산 588 | AWS | 춘천 | 2015~2025 | O | 춘천시 남산면 |

**ASOS**(종관기상관측)는 항목이 많고(일사, 일조 등) 역사가 길다. **AWS**(방재기상관측)는 기본 항목만 있지만 촘촘하게 설치돼 있다.

#### 안내 문서에 적힌 함정 (그대로 반영)
1. **빈칸이 두 가지 뜻**: 강수량 빈칸 = 비가 안 옴 -> **0으로 채움**. 기온이나 습도 빈칸 = 진짜 결측 -> **절대 0으로 채우면 안 됨**
2. 1970~80년대 시간자료는 3시간 간격 관측이라 빈칸이 많음 -> 시간 단위 분석은 2000년 이후만
3. 일자료의 `1시간최다강수량`(이동 구간 최댓값)과 시간자료의 강수량 최댓값(정시 구간)은 다른 값 -> 섞어 쓰지 않기
4. 시간강수량은 **직전 1시간 합계** (18:00 값 = 17:01~18:00에 내린 양), 기온 등은 그 시각의 순간값

### 5-1. 읽기
`통합자료/` 폴더의 지점별 파일(일자료, 시간자료)을 읽는다.
ASOS와 AWS는 **같은 뜻인데 열 이름이 다른 경우**가 있어서(예: `기온(C)`과 `정시기온(C)`), 바꿀 때 같은 새 이름(`temp`)으로 통일한다.
AWS에 없는 열(일사, 일조 등)은 이어 붙일 때 자동으로 빈 값이 된다.

In [22]:
# 지점 정보: {지점번호: (지점명, 종류, 지역)}
STATIONS = {
    108: ('서울', 'ASOS', '강남'), 400: ('강남', 'AWS', '강남'), 401: ('서초', 'AWS', '강남'), 403: ('송파', 'AWS', '강남'),
    101: ('춘천', 'ASOS', '춘천'), 93: ('북춘천', 'ASOS', '춘천'), 675: ('남이섬', 'AWS', '춘천'), 588: ('남산', 'AWS', '춘천'),
}

# 열 이름 바꾸기 규칙. ASOS와 AWS에서 같은 뜻인데 이름이 다른 열은 같은 새 이름으로 통일한다.
DAILY_COLS = {
    '일자': 'date', '지점번호': 'stn_id',
    '일강수량(mm)': 'rain', '1시간최다강수량(mm)': 'rain_1h_max',
    '최고기온(C)': 'tmax', '최저기온(C)': 'tmin', '평균기온(C)': 'tavg',
    '평균습도(%)': 'hum', '최저습도(%)': 'hum_min',           # 최저습도는 ASOS만 있음
    '평균풍속(m/s)': 'wind',
    '일사합(MJ/m2)': 'solar', '일조합(hr)': 'sunshine',       # ASOS만 있음
    '관측시간수': 'obs_hours',                                # AWS만 있음
    '결측여부': 'missing_flag',
}
HOURLY_COLS = {
    '일시': 'datetime', '지점번호': 'stn_id',
    '기온(C)': 'temp', '정시기온(C)': 'temp',                  # ASOS / AWS
    '시간강수량(mm)': 'rain', '1시간강수량(mm)': 'rain',       # ASOS / AWS
    '습도(%)': 'hum', '풍속(m/s)': 'wind',
    '일사(MJ/m2)': 'solar',                                   # ASOS만 있음
}

daily_parts, hourly_parts = [], []
with zipfile.ZipFile(SRC['weather']) as zf:
    files = zip_files(zf)
    merged = {n: i for n, i in files.items() if n.startswith('기상자료/통합자료/')}
    for name, info in sorted(merged.items()):
        df = pd.read_csv(zf.open(info), encoding='utf-8-sig', low_memory=False)
        rule = DAILY_COLS if '일자료' in name else HOURLY_COLS
        df = df[[c for c in rule if c in df.columns]].rename(columns=rule)   # 필요한 열만 골라서 이름 바꾸기
        (daily_parts if '일자료' in name else hourly_parts).append(df)
        print(f"{name.split('/')[-1]:28s} {df.shape}")

wx_daily = pd.concat(daily_parts, ignore_index=True)      # AWS에 없는 열은 자동으로 빈 값이 됨
wx_hourly = pd.concat(hourly_parts, ignore_index=True)

강남_400_시간자료_통합.csv           (48414, 6)
강남_400_일자료_통합.csv            (2024, 11)
남산_588_시간자료_통합.csv           (48411, 6)
남산_588_일자료_통합.csv            (2024, 11)
남이섬_675_시간자료_통합.csv          (48340, 6)
남이섬_675_일자료_통합.csv           (2024, 11)
북춘천_93_일자료_통합.csv            (1748, 13)


서울_108_시간자료_통합.csv           (225215, 7)
서울_108_일자료_통합.csv            (9384, 13)
서초_401_시간자료_통합.csv           (48521, 6)
서초_401_일자료_통합.csv            (2024, 11)
송파_403_시간자료_통합.csv           (48505, 6)
송파_403_일자료_통합.csv            (2024, 11)


춘천_101_시간자료_통합.csv           (225216, 7)
춘천_101_일자료_통합.csv            (9383, 13)


### 5-2. 지점 정보 붙이기와 강수량 빈칸 채우기
- 지점번호만 있으면 알아보기 어려워서 지점명, 종류(ASOS/AWS), 지역 열을 붙인다.
- 강수량 빈칸은 "비가 안 옴"이므로 0으로 채운다 (함정 1).
- 단, **같은 날(시각)의 기온도 비어 있으면** 그때는 관측 자체가 없었던 것이라 0으로 채우지 않고 빈칸으로 둔다.

In [23]:
def add_station_info(df, time_col):
    """지점명, 종류, 지역 열을 붙이고, 시간 열을 날짜형으로 바꾼 뒤, 보기 좋게 정렬한다."""
    df[time_col] = pd.to_datetime(df[time_col])
    df['stn_name'] = df['stn_id'].map(lambda s: STATIONS[s][0])
    df['stn_type'] = df['stn_id'].map(lambda s: STATIONS[s][1])
    df['region'] = df['stn_id'].map(lambda s: STATIONS[s][2])
    front = ['region', 'stn_id', 'stn_name', 'stn_type', time_col]
    df = df[front + [c for c in df.columns if c not in front]]
    return df.sort_values(['stn_id', time_col]).reset_index(drop=True)   # 파일이 최신->과거 순이라 과거->최신으로 다시 정렬


def fill_rain(df, rain_cols, ref_col):
    """강수량 빈칸을 0으로 채운다. (함정 1)
    단, 같은 날(시각)의 기온(ref_col)도 비어 있으면 관측 자체가 없었던 것이므로 빈칸으로 둔다."""
    observed = df[ref_col].notna()
    for c in rain_cols:
        n = (df[c].isna() & observed).sum()
        df.loc[observed, c] = df.loc[observed, c].fillna(0)
        print(f'  {c}: 빈칸 {n:,}개 → 0')
    return df


wx_daily = add_station_info(wx_daily, 'date')
wx_hourly = add_station_info(wx_hourly, 'datetime')
print('일자료'); wx_daily = fill_rain(wx_daily, ['rain', 'rain_1h_max'], ref_col='tmax')
print('시간자료'); wx_hourly = fill_rain(wx_hourly, ['rain'], ref_col='temp')
# 일사량(solar)은 밤에 빈칸인데, 옛날 자료에서는 관측을 안 해서 빈칸인 경우도 섞여 있어 채우지 않는다.

일자료
  rain: 빈칸 12,201개 → 0
  rain_1h_max: 빈칸 14,300개 → 0
시간자료
  rain: 빈칸 365,845개 → 0


### 5-3. 점검: 중복과 분석 기간
분석 기간(2025년 7~12월)에 일자료는 184일, 시간자료는 184 × 24 = 4,416시간이 모두 있어야 한다.
-> 일자료는 모든 지점이 184일 다 있고, 시간자료는 강남, 송파, 남산 지점에서 몇십 시간씩 빠져 있다. 시간 단위로 분석할 때만 주의하면 된다.

In [ ]:
# 점검 1: 같은 지점, 같은 시각이 두 번 나오지 않는가
print('일자료 중복:', wx_daily.duplicated(['stn_id', 'date']).sum(),
      '| 시간자료 중복:', wx_hourly.duplicated(['stn_id', 'datetime']).sum())

# 점검 2: 분석 기간(2025년 7~12월)에 빠진 날이나 시간이 있는가
d = wx_daily[wx_daily['date'].between(START, END)]
check_d = d.groupby(['region', 'stn_name']).agg(
    일수=('date', 'nunique'),                          # 184여야 함
    최고기온_결측=('tmax', lambda s: s.isna().sum()),   # lambda = 이 코드블록에서만 쓰이는 임시함수
    강수_결측=('rain', lambda s: s.isna().sum()),
)
h = wx_hourly[wx_hourly['datetime'].between(START, END + pd.Timedelta(hours=23))]
check_h = h.groupby(['region', 'stn_name']).agg(
    시간수=('datetime', 'nunique'),                    # 184 × 24 = 4,416이어야 함
    기온_결측=('temp', lambda s: s.isna().sum()),
)

# 시간자료가 4,416시간보다 적은 지점은 기록해 둔다
short = check_h[check_h['시간수'] < len(ALL_DAYS) * 24]
lost = {stn: int(len(ALL_DAYS) * 24 - n) for (_, stn), n in short['시간수'].items()}
log_issue('기상_시간자료', f'2025년 하반기 시간자료가 빠진 지점과 빠진 시간 수: {lost}',
          '일자료는 184일 모두 있음. 시간 단위로 분석할 때만 주의')
check_d.join(check_h)    # 두 점검표를 옆으로 붙여서 보기 (북춘천은 시간자료가 없어서 빈칸)

일자료 중복: 0 | 시간자료 중복: 0
[문제 기록] 기상_시간자료: 2025년 하반기 시간자료가 빠진 지점과 빠진 시간 수: {'강남': 33, '송파': 19, '남산': 35} → 일자료는 184일 모두 있음. 시간 단위로 분석할 때만 주의


일수  최고기온_결측  강수_결측     시간수  기온_결측
region stn_name                                    
강남     강남        184        0      0  4383.0    1.0
       서울        184        0      0  4416.0    0.0
       서초        184        0      0  4416.0    0.0
       송파        184        0      0  4397.0    1.0
춘천     남산        184        0      0  4381.0    1.0
       남이섬       184        0      0  4416.0    1.0
       북춘천       184        0      0     NaN    NaN
       춘천        184        0      0  4416.0    1.0

### 5-4. 점검: 전체 기간에서 빠진 날과 옛날 시간자료
평년값(여러 해 평균)을 계산할 때 쓰는 **과거 자료**가 얼마나 비어 있는지 본다.
-> ASOS 시간자료는 1970~80년대에 기온이 40% 가까이 비어 있다(3시간 간격 관측). 안내 문서의 함정 2와 같으므로, **시간 단위 분석은 2000년 이후만** 쓴다.

In [ ]:
#  점검 3: 전체 기간에서 빠진 날 
# (가) 행 자체가 없는 날과 (나) 행은 있는데 최고기온이 빈 날을 따로 센다.
for stn, g in wx_daily.groupby('stn_name'):
    years = range(g['date'].dt.year.min(), END.year + 1)
    expected = pd.DatetimeIndex([d for y in years for d in pd.date_range(f'{y}-07-01', f'{y}-12-31')])  # 매년 7~12월 모든 날짜
    expected = expected[expected >= g['date'].min()]       # 관측 시작일 이전은 빼기 (북춘천은 2016년 10월부터)
    no_row = expected.difference(g['date'])
    no_tmax = g.loc[g['tmax'].isna(), 'date']
    print(f"{stn:4s} 행이 없는 날 {len(no_row)}일 {[d.strftime('%Y-%m-%d') for d in no_row[:3]]}"
          f" / 최고기온이 빈 날 {len(no_tmax)}일 {[d.strftime('%Y-%m-%d') for d in no_tmax[:3]]}")

#  점검 4: 연도별 시간자료 기온 결측률 (함정 2 확인) 
asos_h = wx_hourly[wx_hourly['stn_type'].eq('ASOS')]
miss_by_year = asos_h.groupby(asos_h['datetime'].dt.year)['temp'].apply(lambda s: s.isna().mean())
print('\nASOS 시간자료 기온 결측률이 1%를 넘는 연도:')
display(miss_by_year[miss_by_year > 0.01].map('{:.0%}'.format).rename('기온 결측률').to_frame())

강남   행이 없는 날 0일 [] / 최고기온이 빈 날 11일 ['2015-08-15', '2015-08-16', '2019-08-31']
남산   행이 없는 날 0일 [] / 최고기온이 빈 날 0일 []
남이섬  행이 없는 날 0일 [] / 최고기온이 빈 날 16일 ['2020-08-07', '2020-08-08', '2020-08-09']
북춘천  행이 없는 날 0일 [] / 최고기온이 빈 날 0일 []
서울   행이 없는 날 0일 [] / 최고기온이 빈 날 1일 ['2017-10-12']
서초   행이 없는 날 0일 [] / 최고기온이 빈 날 0일 []
송파   행이 없는 날 0일 [] / 최고기온이 빈 날 0일 []
춘천   행이 없는 날 1일 ['2024-12-25'] / 최고기온이 빈 날 0일 []

ASOS 시간자료 기온 결측률이 1%를 넘는 연도:


,기온 결측률
datetime,
1975,45%
1976,40%
1977,37%
1978,39%
1979,37%
1980,38%
1981,38%
1982,10%
1983,13%


### 5-5. 저장

In [26]:
save(wx_daily, 'wx_daily')
save(wx_hourly, 'wx_hourly')

[저장] wx_daily.parquet  30,635행 × 17열, 0.4MB


[저장] wx_hourly.parquet  692,622행 × 10열, 7.6MB


## 6. 기상특보
폭염, 호우, 한파, 대설 등 **특보 통보문 원문**이다. 한 통보문 안에 여러 특보가 `(1)`, `(2)`처럼 이어져 있어서 파싱(글자를 쪼개 정보를 뽑는 작업)이 필요한다.
파싱은 04번 노트북에서 하고, 여기서는 **읽고 열 이름만 정리해서 저장**한다.

| 원래 열 | 새 이름 | 내용 |
|---|---|---|
| 발표시간 | `issued_at` | 통보문 발표 시각 |
| 지역 | `area_group` | 발표 권역 (서울과 인천, 경기를 묶은 권역이나 강원특별자치도, 전국 등) |
| 발효시각 | `effective_raw` | `(1) 폭염경보 변경 : 2025년 07월 01일 10시 00분` 형식 원문 |
| 해당지역 | `target_raw` | `(1) 폭염경보 변경 : 강원도(동해평지. 양양평지)` 형식 원문 |
| 내용 | `content_raw` | 해제 예고 등 부가 내용 (대부분 빈칸) |

### 6-1. 읽고 열 이름 정리
통보문 원문을 읽고 열 이름만 바꾼다. 특보 종류는 미리보기로 개수만 세어 본다.

In [27]:
text, enc = decode(SRC['warning'].read_bytes())          # read_bytes(): 파일 전체를 바이트로 읽기
wx_warning = pd.read_csv(io.StringIO(text), dtype=str)   # io.StringIO: 글자를 파일처럼 만들어 read_csv에 넘김
wx_warning = wx_warning.rename(columns={'발표시간': 'issued_at', '지역': 'area_group', '발효시각': 'effective_raw',
                                        '해당지역': 'target_raw', '내용': 'content_raw'})
wx_warning['issued_at'] = pd.to_datetime(wx_warning['issued_at'])

print(f'인코딩: {enc} / 크기: {wx_warning.shape}')
print('발표 기간:', wx_warning['issued_at'].min(), '~', wx_warning['issued_at'].max())
print('완전히 같은 행:', wx_warning.duplicated().sum())
print('\n권역별 통보문 수:')
display(wx_warning['area_group'].value_counts().rename('통보문 수').to_frame())

# 특보 종류 미리보기: 'OO주의보' 또는 'OO경보' 글자를 모두 찾아서 셈
# \w+ = 글자 여러 개, (?:주의보|경보) = '주의보' 또는 '경보'
kinds = wx_warning['effective_raw'].str.findall(r'\w+(?:주의보|경보)').explode().value_counts()
print('\n특보 종류별 등장 횟수:')
display(kinds.rename('등장 횟수').to_frame())

인코딩: cp949 / 크기: (4430, 5)
발표 기간: 2025-07-01 10:00:00 ~ 2025-12-31 22:30:00
완전히 같은 행: 0

권역별 통보문 수:


,통보문 수
area_group,
전국,1608
전남광주통합특별시,471
제주도,408
대구·경상북도,380
강원특별자치도,360
서울·인천·경기도,352
부산·울산·경상남도,320
대전·세종·충청남도,249
전북특별자치도,207



특보 종류별 등장 횟수:


,등장 횟수
effective_raw,
풍랑주의보,1762
호우주의보,1654
강풍주의보,844
호우경보,539
폭염주의보,402
폭염경보,225
대설주의보,218
한파주의보,111
폭풍해일주의보,101


### 6-2. 강남, 춘천 관련 통보문 확인과 저장
해당지역 글자에 `서울`, `춘천`이 들어간 통보문이 있는지만 간단히 본다. 정확히 어느 특보가 언제 발효됐는지는 04번에서 파싱해서 확인한다.

In [28]:
# 강남, 춘천 관련 통보문이 있는지 간단히 확인 (정확한 매칭은 04번에서)
for word in ['서울', '춘천']:
    n = wx_warning['target_raw'].str.contains(word, na=False).sum()
    print(f"해당지역에 '{word}'가 들어간 통보문: {n}건")
log_issue('기상특보', '특보 종류, 행위(발표/해제 등), 구역이 한 문자열에 (1),(2)... 로 이어진 원문', '04번에서 항목별로 쪼개서 파싱')
save(wx_warning, 'wx_warning_raw')

해당지역에 '서울'가 들어간 통보문: 114건
해당지역에 '춘천'가 들어간 통보문: 74건
[문제 기록] 기상특보: 특보 종류, 행위(발표/해제 등), 구역이 한 문자열에 (1),(2)... 로 이어진 원문 → 04번에서 항목별로 쪼개서 파싱
[저장] wx_warning_raw.parquet  4,430행 × 5열, 0.2MB


## 7. 지하철 승하차
**날짜 × 노선 × 역**별 승차와 하차 인원이다(2023-01 ~ 2026-08). SK 유동인구는 월 단위라서, **날짜별 사람 흐름은 이 데이터**로 본다.
강남구 역들과 경춘선 춘천권 역들이 모두 들어 있다.

| 원래 열 | 새 이름 |
|---|---|
| 사용일자 | `date` |
| 노선명 | `line` |
| 역명 | `station` |
| 승차총승객수 | `board` (탄 사람) |
| 하차총승객수 | `alight` (내린 사람) |

**주의할 점 2가지**
- 대부분 파일은 각 행 끝에 **쉼표가 하나 더** 붙어 있어서 그냥 읽으면 열이 한 칸씩 밀린다 -> `index_col=False`로 해결
- 44개 중 2개 파일은 인코딩(cp949)과 형식이 다르다 -> `decode` 함수로 자동 처리

### 7-1. 읽기
월별 파일 44개를 읽어서 하나로 이어 붙인다. 파일마다 인코딩이 다를 수 있어서 `decode` 함수로 하나씩 판별한다.

In [29]:
parts, enc_by_file = [], {}
with zipfile.ZipFile(SRC['subway']) as zf:
    for name, info in sorted(zip_files(zf).items()):
        text, enc = decode(zf.read(info))
        # index_col=False: "첫 열을 행 이름으로 쓰지 마라" -> 행 끝 여분의 쉼표 때문에 열이 밀리는 것을 막아줌
        df = pd.read_csv(io.StringIO(text), index_col=False, dtype=str)
        parts.append(df)
        enc_by_file[name[-10:-4]] = enc      # 파일 이름 끝의 '202301' 부분만 기록

odd = [ym for ym, e in enc_by_file.items() if e != 'utf-8-sig']
log_issue('지하철', f'{len(enc_by_file)}개 파일 중 {odd}는 cp949 인코딩 (README에는 utf-8-sig로 적혀 있음)', '파일별 자동 판별')

subway = pd.concat(parts, ignore_index=True)
subway = subway.rename(columns={'사용일자': 'date', '노선명': 'line', '역명': 'station',
                                '승차총승객수': 'board', '하차총승객수': 'alight'})
subway = subway[['date', 'line', 'station', 'board', 'alight']]     # 등록일자는 필요 없어서 뺌
subway['date'] = pd.to_datetime(subway['date'], format='%Y%m%d')
subway['board'] = pd.to_numeric(subway['board']).astype('int64')
subway['alight'] = pd.to_numeric(subway['alight']).astype('int64')
subway.head()

[문제 기록] 지하철: 44개 파일 중 ['202402', '202502']는 cp949 인코딩 (README에는 utf-8-sig로 적혀 있음) → 파일별 자동 판별


,date,line,station,board,alight
0,2023-01-01,경강선,곤지암,1450,1539
1,2023-01-01,2호선,신도림,20210,21037
2,2023-01-01,2호선,문래,5995,6567
3,2023-01-01,2호선,영등포구청,5629,6117
4,2023-01-01,2호선,당산,6821,7594


### 7-2. 강남, 춘천 역 표시
강남구와 춘천권 역에 `region` 표시를 붙인다. 역마다 주소를 기준으로 **강남구 안에 있는 역**만 넣었다.
- 제외: `새절(신사)`(6호선, 은평구), `인천논현`(인천), 교대, 양재(서초구)
- 춘천권: `굴봉산`은 경기도 가평군 소재로 보여 제외했다. **03번에서 역 주소를 한 번 더 확인한다.**
- 역 목록은 03번 노트북에서 다시 검토할 수 있도록 여기 한곳에만 적어둔다.
- 신분당선은 집계되지 않았다.

In [30]:
GANGNAM_STATIONS = [
    '강남', '역삼', '선릉', '삼성(무역센터)',                                  # 2호선
    '신사', '압구정', '도곡', '대치', '학여울', '대청', '일원', '수서', '매봉',   # 3호선
    '논현', '학동', '강남구청', '청담',                                        # 7호선
    '신논현', '언주', '선정릉', '삼성중앙', '봉은사',                            # 9호선
    '압구정로데오', '한티', '구룡', '개포동', '대모산입구',                       # 수인분당선
]
CHUNCHEON_STATIONS = ['춘천', '남춘천', '김유정', '강촌', '백양리']               # 경춘선

# 목록에 적은 역 이름이 실제 데이터에 있는지 확인 (오타 방지)
all_stations = set(subway['station'])
for s in GANGNAM_STATIONS + CHUNCHEON_STATIONS:
    assert s in all_stations, f"'{s}' 역이 데이터에 없다."

subway['region'] = np.select(                     # np.select: 조건 목록에 따라 값 고르기 (엑셀의 중첩 IF와 비슷)
    [subway['station'].isin(GANGNAM_STATIONS), subway['station'].isin(CHUNCHEON_STATIONS)],
    ['강남', '춘천'],
    default='기타',
)
# 지역별로 어떤 '노선 역'이 들어갔는지 확인 (환승역은 노선별로 따로 나옴)
tagged = subway.loc[subway['region'] != '기타', ['region', 'line', 'station']].drop_duplicates()
for region, g in tagged.groupby('region'):
    print(f"{region} ({len(g)}개): {', '.join(sorted(g['line'] + ' ' + g['station']))}")

강남 (32개): 2호선 강남, 2호선 삼성(무역센터), 2호선 선릉, 2호선 역삼, 3호선 대청, 3호선 대치, 3호선 도곡, 3호선 매봉, 3호선 수서, 3호선 신사, 3호선 압구정, 3호선 일원, 3호선 학여울, 7호선 강남구청, 7호선 논현, 7호선 청담, 7호선 학동, 9호선 신논현, 9호선2~3단계 봉은사, 9호선2~3단계 삼성중앙, 9호선2~3단계 선정릉, 9호선2~3단계 언주, 분당선 강남구청, 분당선 개포동, 분당선 구룡, 분당선 대모산입구, 분당선 도곡, 분당선 선릉, 분당선 선정릉, 분당선 수서, 분당선 압구정로데오, 분당선 한티
춘천 (5개): 경춘선 강촌, 경춘선 김유정, 경춘선 남춘천, 경춘선 백양리, 경춘선 춘천


### 7-3. 기본 점검
- 전체 기간에 빠진 날짜, 같은 날 같은 역이 두 번 나오는 행, 음수 값이 없는지 확인한다.
- 분석 기간 184일 동안 강남, 춘천 역이 **하루도 빠짐없이** 있는지 확인한다.

In [31]:
print('기간:', subway['date'].min().date(), '~', subway['date'].max().date())
expected = pd.date_range(subway['date'].min(), subway['date'].max())
print('빠진 날짜:', list(expected.difference(subway['date'].unique())))
print('날짜, 노선, 역 중복:', subway.duplicated(['date', 'line', 'station']).sum())
print('음수 값:', (subway[['board', 'alight']] < 0).sum().sum())

# 분석 기간 동안 강남, 춘천 역이 매일 빠짐없이 있는가? (환승역은 노선별로 따로 행이 있어서 노선별 역 단위로 봄)
win = subway[subway['date'].between(START, END) & (subway['region'] != '기타')]
days = win.groupby(['region', 'line', 'station'])['date'].nunique()
print(f'\n분석 기간 중 184일이 안 되는 노선별 역: {(days < len(ALL_DAYS)).sum()}개')
display(days[days < len(ALL_DAYS)].rename('날짜 수').to_frame())

기간: 2023-01-01 ~ 2026-08-31
빠진 날짜: []
날짜, 노선, 역 중복: 0
음수 값: 0

분석 기간 중 184일이 안 되는 노선별 역: 0개


,,,날짜 수
region,line,station,


### 7-4. 저장

In [32]:
save(subway, 'subway')

[저장] subway.parquet  822,891행 × 6열, 4.4MB


## 8. 서울 상권분석 추정매출 (행정동, 분기)
서울시가 공개한 **행정동 × 업종(63개) × 분기**별 추정 매출이다(2021~2025). 요일, 시간대, 성별, 연령별로 나뉜 금액과 건수가 있다.
**춘천은 없고**, 분기 단위라 날씨 이벤트 분석에는 못 쓴다. 06번에서 **장기 추세와 카드 데이터 대표성 검증**에 쓴다.

- 열이 53개라서 이름은 원래 한글 그대로 둔다.
- 연도별 zip 5개가 큰 zip 안에 들어 있는 **중첩 zip**이다.
- 건수 쪽 시간대 열 이름이 깨져 있어서(`시간대_건수~06_매출_건수`) 금액 쪽과 같은 형식(`시간대_00~06_매출_건수`)으로 고친다.
- 시간대 구간이 카드 데이터와 다르다: 00~06 / 06~11 / 11~14 / 14~17 / 17~21 / 21~24

### 8-1. 읽기
큰 zip 안의 연도별 zip을 차례로 열어서 읽고, 깨진 열 이름을 고친다.
분기 코드(`20251`)를 연도(`year`)와 분기(`quarter`)로 나누고, 강남구 행정동에는 `is_gangnam=True` 표시를 붙인다.

In [33]:
# 깨진 열 이름 -> 올바른 이름. {'시간대_건수~06_매출_건수': '시간대_00~06_매출_건수', ...}
TIME_BANDS = {'06': '00~06', '11': '06~11', '14': '11~14', '17': '14~17', '21': '17~21', '24': '21~24'}
FIX_COLS = {f'시간대_건수~{end}_매출_건수': f'시간대_{band}_매출_건수' for end, band in TIME_BANDS.items()}

parts = []
with zipfile.ZipFile(SRC['sales']) as outer:
    for oname, oinfo in sorted(zip_files(outer).items()):
        inner = zipfile.ZipFile(io.BytesIO(outer.read(oinfo)))   # zip 안의 zip -> 메모리에 올려서 한 번 더 열기
        for iname, iinfo in zip_files(inner).items():
            text, enc = decode(inner.read(iinfo))
            df = pd.read_csv(io.StringIO(text), dtype={'행정동_코드': str, '서비스_업종_코드': str})
            print(f'{iname}: {df.shape} ({enc})')
            parts.append(df)

sales = pd.concat(parts, ignore_index=True).rename(columns=FIX_COLS)
log_issue('상권매출', "시간대별 건수 열 이름이 깨져 있음 ('시간대_건수~06_매출_건수' 등)", '금액 열과 같은 형식으로 수정')

sales['year'] = sales['기준_년분기_코드'] // 10          # 20251 -> 2025  ( // 는 몫)
sales['quarter'] = sales['기준_년분기_코드'] % 10        # 20251 -> 1     ( % 는 나머지)
sales['is_gangnam'] = sales['행정동_코드'].str.startswith('11680')   # 강남구 행정동 코드는 11680으로 시작
print('\n분기 목록:', sorted(sales['기준_년분기_코드'].unique()))
print('강남구 행정동 수(연도별):', sales[sales['is_gangnam']].groupby('year')['행정동_코드'].nunique().to_dict())

서울시 상권분석서비스(추정매출-행정동)_2024년.csv: (67900, 53) (cp949)


서울시 상권분석서비스(추정매출-행정동)_2025년.csv: (67113, 53) (cp949)


서울시_상권분석서비스(추정매출-행정동)_2021년.csv: (70071, 53) (cp949)


서울시_상권분석서비스(추정매출-행정동)_2022년.csv: (69440, 53) (cp949)


서울시_상권분석서비스(추정매출-행정동)_2023년.csv: (68643, 53) (cp949)
[문제 기록] 상권매출: 시간대별 건수 열 이름이 깨져 있음 ('시간대_건수~06_매출_건수' 등) → 금액 열과 같은 형식으로 수정

분기 목록: [20211, 20212, 20213, 20214, 20221, 20222, 20223, 20224, 20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244, 20251, 20252, 20253, 20254]
강남구 행정동 수(연도별): {2021: 22, 2022: 22, 2023: 22, 2024: 22, 2025: 22}


### 8-2. 점검: 쪼갠 금액을 더하면 전체와 같은가?
요일별 금액 7개를 더하면 당월 매출(전체)과 같아야 한다. 시간대, 성별, 연령대도 마찬가지이다.
-> 요일과 시간대는 100% 맞지만, 성별과 연령대는 합이 전체보다 적다. <br>성별이나 연령을 알 수 없는 매출(법인 등)이 빠진 것으로 보인다.<br>
- 행마다 비율을 낸 뒤 **중앙값**을 보면 약 98.5%라 거의 다 맞는 것처럼 보이지만,
- **금액 기준**(성별 합계 전체 / 당월 매출 전체)으로 보면 약 **89%** (강남만 약 88%) 이다. 매출이 큰 행에서 더 많이 빠진다는 뜻이다.
- 행의 절반 정도(53%)가 1% 넘게 어긋난다.
- 아마도 법인카드와 사업자가 구매한 데이터는 제외한 것으로 보인다.<br>
그래서 성별, 연령 비중을 낼 때는 당월 매출이 아니라 **성별(연령) 합계를 분모**로 써야 한다.

In [ ]:
#  점검: 쪼갠 금액을 다시 더하면 전체(당월_매출_금액)와 같은가? 
print('분기, 행정동, 업종 중복:', sales.duplicated(['기준_년분기_코드', '행정동_코드', '서비스_업종_코드']).sum())

total = sales['당월_매출_금액']
splits = {
    '요일':      [f'{d}요일_매출_금액' for d in '월화수목금토일'],
    '주중+주말': ['주중_매출_금액', '주말_매출_금액'],
    '시간대':    [f'시간대_{b}_매출_금액' for b in TIME_BANDS.values()],
    '성별':      ['남성_매출_금액', '여성_매출_금액'],
    '연령대':    ['연령대_10_매출_금액', '연령대_20_매출_금액', '연령대_30_매출_금액',
                 '연령대_40_매출_금액', '연령대_50_매출_금액', '연령대_60_이상_매출_금액'],
}
ok = total > 0     # 전체가 0인 행은 나눗셈이 안 되므로 뺌
medians, weighted = {}, {}
for name, cols in splits.items():
    ratio = sales.loc[ok, cols].sum(axis=1) / total[ok]
    off = (ratio - 1).abs() > 0.01                        # 1% 넘게 어긋나는 행
    medians[name] = ratio.median()
    weighted[name] = sales.loc[ok, cols].sum().sum() / total[ok].sum()   # 금액 기준: 모든 행을 합친 뒤 나눔
    print(f'{name:8s} 합계/전체: 중앙값 {ratio.median():.4f}, 금액 기준 {weighted[name]:.4f}, 1% 넘게 어긋난 행 {off.mean():.2%}')

log_issue('상권매출', f"성별, 연령대 금액을 더하면 금액 기준 당월 매출의 약 {weighted['성별']:.0%}뿐임 (행별 중앙값은 {medians['성별']:.1%}. 요일, 시간대는 100%)",
          '성별이나 연령을 알 수 없는 매출(법인 등)로 추정. 성별, 연령 비중을 낼 때는 성별(연령) 합계를 분모로 쓰기')

분기, 행정동, 업종 중복: 0
요일       합계/전체: 중앙값 1.0000, 금액 기준 1.0000, 1% 넘게 어긋난 행 0.00%
주중+주말    합계/전체: 중앙값 1.0000, 금액 기준 1.0000, 1% 넘게 어긋난 행 0.00%
시간대      합계/전체: 중앙값 1.0000, 금액 기준 1.0000, 1% 넘게 어긋난 행 0.00%
성별       합계/전체: 중앙값 0.9852, 금액 기준 0.8922, 1% 넘게 어긋난 행 53.08%
연령대      합계/전체: 중앙값 0.9852, 금액 기준 0.8922, 1% 넘게 어긋난 행 53.08%
[문제 기록] 상권매출: 성별, 연령대 금액을 더하면 금액 기준 당월 매출의 약 89%뿐임 (행별 중앙값은 98.5%. 요일, 시간대는 100%) → 성별이나 연령을 알 수 없는 매출(법인 등)로 추정. 성별, 연령 비중을 낼 때는 성별(연령) 합계를 분모로 쓰기


### 8-3. 저장

In [35]:
save(sales, 'sales_seoul')

[저장] sales_seoul.parquet  343,167행 × 56열, 72.9MB


## 9. 상가(상권) 정보
소상공인시장진흥공단의 **상가 업소 목록**이다. 강남구와 춘천시 각각 **2025년 10월, 12월 두 시점**의 스냅샷이다.
두 시점에 모두 있는 업소, 한쪽에만 있는 업소를 비교하면 **개업과 폐업**을 볼 수 있다(06번).

원래 39개 열 중 분석에 필요한 열만 남길다(지번 코드, 우편번호 등은 뺌).

### 9-1. 읽기
파일 4개(강남구, 춘천시 × 10월, 12월)를 읽고, 지역(`region`)과 시점(`snapshot`) 열을 맨 앞에 붙인다.

In [36]:
STORE_COLS = ['상가업소번호', '상호명', '지점명',
              '상권업종대분류명', '상권업종중분류명', '상권업종소분류명', '표준산업분류명',
              '시군구명', '행정동코드', '행정동명', '도로명주소', '층정보', '경도', '위도']

parts = []
with zipfile.ZipFile(SRC['stores']) as zf:
    for name, info in sorted(zip_files(zf).items()):
        region = '강남' if name.startswith('강남구') else '춘천'
        snapshot = re.search(r'(\d{6})', name).group(1)      # 파일 이름에서 '202510' 같은 숫자 6자리 뽑기
        df = pd.read_csv(zf.open(info), encoding='utf-8-sig', dtype=str, usecols=STORE_COLS)  # usecols: 이 열들만 읽기
        df.insert(0, 'region', region)          # 맨 앞(0번째)에 열 추가
        df.insert(1, 'snapshot', snapshot)
        parts.append(df)
        print(f'{name}: {len(df):,}개 업소')

stores = pd.concat(parts, ignore_index=True)
stores['경도'] = pd.to_numeric(stores['경도'])
stores['위도'] = pd.to_numeric(stores['위도'])

강남구_202510.csv: 64,146개 업소


강남구_202512.csv: 64,123개 업소
춘천시_202510.csv: 17,497개 업소
춘천시_202512.csv: 17,486개 업소


### 9-2. 점검과 개업, 폐업 미리보기
- 같은 시점에 같은 업소번호가 두 번 나오지 않는지, 좌표가 비어 있지 않은지 확인한다.
- **10월에만 있는 업소 = 폐업 추정**, **12월에만 있는 업소 = 개업 추정** 으로 개수만 미리 본다. 자세한 분석은 06번에서 한다.
- 교수님이 말씀하신 "https://file.localdata.go.kr/file/general_restaurants/info" 행안부 폐업 데이터도 다운받아서 확인해보기 .. (일단은 패쓰)

In [37]:
print('\n시점 안에서 같은 업소번호 중복:', stores.duplicated(['snapshot', '상가업소번호']).sum())
print('좌표 빈 값:', stores[['경도', '위도']].isna().sum().sum())
print('업종 대분류:', sorted(stores['상권업종대분류명'].unique()))

# 두 시점 비교 미리보기: 10월에만 있음(=폐업 추정), 12월에만 있음(=개업 추정)
for region in ['강남', '춘천']:
    ids = {s: set(stores.loc[(stores['region'] == region) & (stores['snapshot'] == s), '상가업소번호'])
           for s in ['202510', '202512']}
    print(f"{region}: 두 시점 모두 {len(ids['202510'] & ids['202512']):,} / "
          f"10월에만 {len(ids['202510'] - ids['202512']):,} / 12월에만 {len(ids['202512'] - ids['202510']):,}")


시점 안에서 같은 업소번호 중복: 0
좌표 빈 값: 0
업종 대분류: ['과학·기술', '교육', '보건의료', '부동산', '소매', '수리·개인', '숙박', '시설관리·임대', '예술·스포츠', '음식']
강남: 두 시점 모두 62,737 / 10월에만 1,409 / 12월에만 1,386
춘천: 두 시점 모두 17,063 / 10월에만 434 / 12월에만 423


### 9-3. 저장

In [38]:
save(stores, 'stores')

[저장] stores.parquet  163,252행 × 16열, 6.2MB


## 10. 관광 방문자 (한국관광 데이터랩)
강남구와 춘천시의 **월별 방문자 수**이다(2021~2025). 큰 zip 안에 "지역 × 내국인/외국인 × 연도" zip이 20개 있고, 각각에 CSV가 3개씩 들어 있다.
CSV 종류별로 모아서 표 6개로 저장한다.

| 저장 이름 | 내용 | 단위 |
|---|---|---|
| `tour_dom_trend` | 내국인 방문자 수 추이 (현지인 / 외지인 / 전체) | 월 |
| `tour_dom_dong` | 내국인 행정동별 방문자 수와 비율 | 연 |
| `tour_dom_home` | 내국인 방문자 거주지(시도, 시군구) 비율 | 연 |
| `tour_for_trend` | 외국인 방문자 수 추이 | 월 |
| `tour_for_dong` | 외국인 행정동별 방문자 수 | 연 |
| `tour_for_country` | 외국인 방문자 국적 비율 | 연 |

**현지인** = 그 지역 거주자, **외지인** = 다른 지역에서 온 사람. 방문자 수는 통신 데이터로 추정한 값이라 사람 수라기보다 "방문 횟수"에 가깝다(한 사람이 여러 번 셀 수 있음).

### 10-1. 읽기
큰 zip -> 안쪽 zip 20개 -> 각각 CSV 3개 순서로 연다.
CSV 이름 앞의 날짜 숫자(`20260918200700_`)를 떼어내면 종류를 알 수 있어서, 종류별로 모아 표 6개를 만든다.

In [39]:
# {안쪽 CSV 이름(앞의 날짜 숫자 뺀 것): (저장 이름, {원래 열: 새 열})}
TOUR_KINDS = {
    '방문자 수 추이.csv':             ('tour_dom_trend',   {'기준년월': 'ym', '기초지자체': 'sgg', '방문자 구분': 'visitor_type', '방문자 수': 'visitors'}),
    '지역별 방문자 수.csv':           ('tour_dom_dong',    {'기초지자체명': 'dong', '기초지자체 방문자 수': 'visitors', '기초지자체 방문자 비율': 'share'}),
    '방문자 거주지.csv':              ('tour_dom_home',    {'거주지(시도)': 'home_sido', '거주지(시군구)': 'home_sgg', '비율(%)': 'share'}),
    '외국인 방문자 수 추이.csv':      ('tour_for_trend',   {'날짜': 'ym', '지역': 'sgg', '외국인 방문자수': 'visitors'}),
    '외국인 지역별 방문자 수.csv':    ('tour_for_dong',    {'지역': 'dong', '외국인 방문자수': 'visitors'}),
    '외국인 방문자 거주지(국가).csv': ('tour_for_country', {'국가명': 'country', '비율(%)': 'share'}),
}
SGG_TO_REGION = {'강남구': '강남', '춘천시': '춘천'}

tour_parts = {out: [] for out, _ in TOUR_KINDS.values()}     # 저장 이름별로 빈 목록 준비
with zipfile.ZipFile(SRC['tour']) as outer:
    for oname, oinfo in sorted(zip_files(outer).items()):
        # 'datalab_강남구_내국인_월간_202501-202512.zip' 에서 ('강남구', '2025') 뽑기
        sgg, year = re.match(r'datalab_(강남구|춘천시)_(?:내국인|외국인)_월간_(\d{4})', oname).groups()
        inner = zipfile.ZipFile(io.BytesIO(outer.read(oinfo)))
        for iname, iinfo in zip_files(inner).items():
            kind = re.sub(r'^\d+_', '', iname)      # '20260918200700_방문자 수 추이.csv' -> '방문자 수 추이.csv'
            out, cols = TOUR_KINDS[kind]
            text, _ = decode(inner.read(iinfo))
            df = pd.read_csv(io.StringIO(text)).rename(columns=cols)
            df.insert(0, 'region', SGG_TO_REGION[sgg])
            df.insert(1, 'year', int(year))
            tour_parts[out].append(df)

tour = {out: pd.concat(parts, ignore_index=True) for out, parts in tour_parts.items()}
for out, df in tour.items():
    print(f'{out:18s} {df.shape}  열: {list(df.columns)}')

tour_dom_trend     (360, 6)  열: ['region', 'year', 'ym', 'sgg', 'visitor_type', 'visitors']
tour_dom_dong      (236, 5)  열: ['region', 'year', 'dong', 'visitors', 'share']
tour_dom_home      (2482, 5)  열: ['region', 'year', 'home_sido', 'home_sgg', 'share']
tour_for_trend     (120, 5)  열: ['region', 'year', 'ym', 'sgg', 'visitors']
tour_for_dong      (235, 4)  열: ['region', 'year', 'dong', 'visitors']
tour_for_country   (238, 4)  열: ['region', 'year', 'country', 'share']


### 10-2. 월별 추이 표 정리와 점검
- `202501` 같은 숫자를 날짜형 `month` 열로 바꾼다.
- 연도별 파일끼리 겹치는 달이 있으면 하나만 남길다.
- **현지인 + 외지인 = 전체** 인지 확인한다. 비율이 모두 1이면 정상이다.

In [40]:
# 월별 추이 표 정리: 202501 -> 날짜형 month 열, 방문자 구분 이름 짧게
VISITOR_TYPE = {'현지인방문자(a)': '현지인', '외지인방문자(b)': '외지인', '전체방문자(a+b)': '전체'}
tour['tour_dom_trend']['visitor_type'] = tour['tour_dom_trend']['visitor_type'].map(VISITOR_TYPE)

for out in ['tour_dom_trend', 'tour_for_trend']:
    df = tour[out]
    df['month'] = pd.to_datetime(df['ym'].astype(str), format='%Y%m')
    key = ['region', 'month'] + (['visitor_type'] if 'visitor_type' in df.columns else [])
    n_dup = df.duplicated(key).sum()
    if n_dup:                                           # 연도별 파일끼리 겹치는 달이 있으면 제거
        log_issue(out, f'연도별 파일끼리 겹치는 달 {n_dup}행', '중복 제거')
        df = df.drop_duplicates(key)
    tour[out] = df
    months = df.groupby('region')['month'].agg(['min', 'max', 'nunique'])
    print(f'{out}: 지역별 기간과 개월 수\n{months}\n')

print('현지인 + 외지인 = 전체 인지 확인 (1이면 정상):')
p = tour['tour_dom_trend'].pivot_table(index=['region', 'month'], columns='visitor_type', values='visitors')
display(((p['현지인'] + p['외지인']) / p['전체']).describe()[['min', 'max']].rename('비율').to_frame())

tour_dom_trend: 지역별 기간과 개월 수
              min        max  nunique
region                               
강남     2021-01-01 2025-12-01       60
춘천     2021-01-01 2025-12-01       60

tour_for_trend: 지역별 기간과 개월 수
              min        max  nunique
region                               
강남     2021-01-01 2025-12-01       60
춘천     2021-01-01 2025-12-01       60

현지인 + 외지인 = 전체 인지 확인 (1이면 정상):


,비율
min,1.0
max,1.0


### 10-3. 저장

In [41]:
for out, df in tour.items():
    save(df, out)

[저장] tour_dom_trend.parquet  360행 × 7열, 0.0MB
[저장] tour_dom_dong.parquet  236행 × 5열, 0.0MB
[저장] tour_dom_home.parquet  2,482행 × 5열, 0.0MB
[저장] tour_for_trend.parquet  120행 × 6열, 0.0MB
[저장] tour_for_dong.parquet  235행 × 4열, 0.0MB
[저장] tour_for_country.parquet  238행 × 4열, 0.0MB


## 11. 행정동 경계와 코드 연결표
전국 행정동 경계 지도에서 **강남구(22개 동)와 춘천시(25개 동)** 만 잘라서 저장한다.

행정동 코드는 **기관마다 체계가 달라서** 데이터를 서로 붙일 때 헷갈린다. 여기서 연결표를 만들어 둔다.

| 열 | 체계 | 예시 (강남구 신사동) | 쓰는 데이터 |
|---|---|---|---|
| `adm_cd` | 통계청 8자리 | `11230510` | SK 유동인구 (`block_cd` 앞 8자리) |
| `adm_cd2` | 행정안전부 10자리 | `1168051000` | 경계 파일 |
| `adm_cd8_mois` | 행정안전부 앞 8자리 | `11680510` | 상권매출(04), 상가정보(06) |

### 11-1. 강남, 춘천 행정동 경계 자르기와 연결표 저장
전국 경계 파일에서 두 지역만 남기고, 세 가지 코드 체계를 한 표(`dong_codes.csv`)에 모아 저장한다.

In [42]:
gdf = gpd.read_file(SRC['boundary'])
print('전체 행정동 수:', len(gdf), '| 좌표계:', gdf.crs)    # 좌표계 = 위도, 경도(WGS84)

dong = gdf[gdf['sggnm'].isin(['강남구', '춘천시'])].copy()
dong['region'] = dong['sggnm'].map({'강남구': '강남', '춘천시': '춘천'})
dong['dong'] = dong['adm_nm'].str.split().str[-1]          # '서울특별시 강남구 신사동' -> '신사동' (띄어쓰기로 나눈 마지막 조각)
dong['adm_cd8_mois'] = dong['adm_cd2'].str[:8]
dong = dong[['region', 'dong', 'adm_nm', 'adm_cd', 'adm_cd2', 'adm_cd8_mois', 'geometry']].reset_index(drop=True)
display(dong.groupby('region').size().rename('행정동 수').to_frame())

dong.to_file(OUT / 'dong_boundary.geojson', driver='GeoJSON')
dong_codes = pd.DataFrame(dong.drop(columns='geometry'))
dong_codes.to_csv(OUT / 'dong_codes.csv', index=False, encoding='utf-8-sig')
for fname, df in [('dong_boundary.geojson', dong), ('dong_codes.csv', dong_codes)]:
    SAVED.append({'파일': fname, '행': len(df), '열': df.shape[1], '크기(MB)': round((OUT / fname).stat().st_size / 1e6, 1)})
dong_codes.head()

전체 행정동 수: 3554 | 좌표계: EPSG:4326


,행정동 수
region,
강남,22
춘천,25


,region,dong,adm_nm,adm_cd,adm_cd2,adm_cd8_mois
0,강남,신사동,서울특별시 강남구 신사동,11230510,1168051000,11680510
1,강남,논현1동,서울특별시 강남구 논현1동,11230520,1168052100,11680521
2,강남,논현2동,서울특별시 강남구 논현2동,11230530,1168053100,11680531
3,강남,삼성1동,서울특별시 강남구 삼성1동,11230580,1168058000,11680580
4,강남,삼성2동,서울특별시 강남구 삼성2동,11230590,1168059000,11680590


### 11-2. 매칭률 확인
연결표로 다른 데이터의 행정동이 **얼마나 잘 매칭되는지** 확인한다. 100%에 가까워야 한다.

In [43]:
def match_rate(values, reference):
    """values 중에서 reference 목록에 있는 값의 비율과, 없는 값(최대 10개)을 돌려준다."""
    values = pd.Series(values)
    ok = values.isin(set(reference))
    return ok.mean(), sorted(values[~ok].unique())[:10]

checks = {
    'SK 유동인구 (adm_cd, 격자 행 기준)':    match_rate(flow['age']['adm_cd'], dong_codes['adm_cd']),
    '상가정보 (adm_cd8_mois, 업소 행 기준)':  match_rate(stores['행정동코드'], dong_codes['adm_cd8_mois']),
    '상권매출 강남 (adm_cd8_mois, 동 기준)':  match_rate(sales.loc[sales['is_gangnam'], '행정동_코드'].unique(), dong_codes['adm_cd8_mois']),
    '관광 내국인 (지역+동 이름, 동 기준)':    match_rate(
        (tour['tour_dom_dong']['region'] + ' ' + tour['tour_dom_dong']['dong']).unique(),
        dong_codes['region'] + ' ' + dong_codes['dong']),
}
for name, (rate, misses) in checks.items():
    print(f'{name:38s} 매칭률 {rate:6.1%}  못 붙은 값: {misses}')

SK 유동인구 (adm_cd, 격자 행 기준)              매칭률  99.5%  못 붙은 값: ['11230740']
상가정보 (adm_cd8_mois, 업소 행 기준)           매칭률 100.0%  못 붙은 값: []
상권매출 강남 (adm_cd8_mois, 동 기준)           매칭률  95.5%  못 붙은 값: ['11680740']
관광 내국인 (지역+동 이름, 동 기준)                 매칭률  97.9%  못 붙은 값: ['강남 일원2동']


### 11-3. 일원2동 -> 개포3동 코드 통일
11-2에서 매칭이 안된 값은 모두 **강남구 일원2동** 하나이다.

일원2동은 **개포3동으로 이름과 코드가 바뀐 동**이다. 관광 데이터를 보면 2023년에는 두 이름이 함께 나오고, 2024년부터는 개포3동만 나온다.
경계 파일(2025년판)은 새 코드를 쓰는데, SK 유동인구와 상권매출은 옛 코드를 그대로 쓴다.

| | 옛날 (일원2동) | 지금 (개포3동) | 옛 코드를 쓰는 데이터 |
|---|---|---|---|
| 통계청 코드 | `11230740` | `11230511` | SK 유동인구 |
| 행안부 코드 | `11680740` | `11680675` | 상권매출 |
| 이름 | 일원2동 | 개포3동 | 상권매출, 관광 |

-> 옛 코드와 이름을 새 것으로 바꿔서 모든 데이터를 경계 파일 기준에 맞춘다.

In [44]:
OLD_TO_NEW = {
    'adm_cd':       {'11230740': '11230511'},   # 통계청 코드 (SK 유동인구)
    'adm_cd8_mois': {'11680740': '11680675'},   # 행안부 코드 (상권매출)
    'dong':         {'일원2동': '개포3동'},       # 이름 (상권매출, 관광)
}

# SK 유동인구: 행정동 코드만 바꿈 (격자 자체는 그대로)
for kind in FLOW_KINDS:
    flow[kind]['adm_cd'] = flow[kind]['adm_cd'].replace(OLD_TO_NEW['adm_cd'])

# 상권매출: 코드와 이름 모두 바꿈
sales['행정동_코드'] = sales['행정동_코드'].replace(OLD_TO_NEW['adm_cd8_mois'])
sales['행정동_코드_명'] = sales['행정동_코드_명'].replace(OLD_TO_NEW['dong'])
assert not sales.duplicated(['기준_년분기_코드', '행정동_코드', '서비스_업종_코드']).any(), '바꾼 뒤 중복이 생겼는다.'

# 관광: 이름을 바꾼 뒤, 2023년처럼 두 이름이 같이 있던 해는 같은 동의 값을 더해서 한 행으로 합침
for out in ['tour_dom_dong', 'tour_for_dong']:
    df = tour[out].copy()
    df['dong'] = df['dong'].replace(OLD_TO_NEW['dong'])
    value_cols = [c for c in ['visitors', 'share'] if c in df.columns]
    tour[out] = df.groupby(['region', 'year', 'dong'], as_index=False)[value_cols].sum()

log_issue('행정동', '일원2동 → 개포3동으로 이름과 코드가 바뀜. 경계 파일은 새 코드, SK, 상권매출, 관광은 옛 코드',
          '옛 코드와 이름을 새 것으로 통일')

# 다시 매칭률 확인 -> 모두 100%여야 함
checks = {
    'SK 유동인구':   match_rate(flow['age']['adm_cd'], dong_codes['adm_cd']),
    '상권매출 강남': match_rate(sales.loc[sales['is_gangnam'], '행정동_코드'].unique(), dong_codes['adm_cd8_mois']),
    '관광 내국인':   match_rate((tour['tour_dom_dong']['region'] + ' ' + tour['tour_dom_dong']['dong']).unique(),
                               dong_codes['region'] + ' ' + dong_codes['dong']),
    '관광 외국인':   match_rate((tour['tour_for_dong']['region'] + ' ' + tour['tour_for_dong']['dong']).unique(),
                               dong_codes['region'] + ' ' + dong_codes['dong']),
}
for name, (rate, misses) in checks.items():
    print(f'{name:10s} 매칭률 {rate:6.1%}  못 붙은 값: {misses}')

[문제 기록] 행정동: 일원2동 → 개포3동으로 이름과 코드가 바뀜. 경계 파일은 새 코드, SK, 상권매출, 관광은 옛 코드 → 옛 코드와 이름을 새 것으로 통일


SK 유동인구    매칭률 100.0%  못 붙은 값: []
상권매출 강남    매칭률 100.0%  못 붙은 값: []
관광 내국인     매칭률 100.0%  못 붙은 값: []
관광 외국인     매칭률 100.0%  못 붙은 값: []


### 11-4. 바꾼 표 다시 저장
4장, 8장, 10장에서 저장한 파일은 옛 코드 그대로다. 코드를 바꾼 표로 **같은 이름에 덮어써서** 저장한다.

In [45]:
# 코드를 바꾼 표들을 다시 저장한다 (같은 이름으로 덮어씀)
for kind in FLOW_KINDS:
    save(flow[kind], f'flow_{kind}')
save(sales, 'sales_seoul')
save(tour['tour_dom_dong'], 'tour_dom_dong')
save(tour['tour_for_dong'], 'tour_for_dong')

[저장] flow_age.parquet  595,915행 × 18열, 11.0MB


[저장] flow_time.parquet  570,475행 × 30열, 16.6MB


[저장] flow_wkdy.parquet  720,165행 × 13열, 11.0MB


[저장] sales_seoul.parquet  343,167행 × 56열, 72.9MB
[저장] tour_dom_dong.parquet  235행 × 5열, 0.0MB
[저장] tour_for_dong.parquet  235행 × 4열, 0.0MB


## 12. 달력
분석 기간 184일의 **요일, 주말, 공휴일, 추석 연휴** 표시이다. 매출은 날씨보다 요일과 연휴의 영향을 훨씬 크게 받아서, 05번에서 이 효과를 빼고 날씨 효과를 본다.

2025년 하반기 공휴일:
- 8/15(금) 광복절
- 10/3(금) 개천절
- 10/5(일)~10/7(화) 추석 연휴
- 10/8(수) 추석 대체공휴일 (추석 연휴가 일요일과 겹쳐서)
- 10/9(목) 한글날
- 12/25(목) 성탄절

-> **10/3(금)~10/9(목)이 7일 연속 쉬는 날**이었다. 10/10(금)은 평일이지만 연차를 쓴 사람이 많았을 수 있다.

### 12-1. 달력 표 만들기와 저장

| 열 | 뜻 |
|---|---|
| `dow`, `dow_name` | 요일 (0=월 ... 6=일) |
| `is_weekend` | 토요일, 일요일 |
| `holiday_name`, `is_holiday` | 공휴일 이름과 여부 |
| `is_offday` | 쉬는 날 = 주말 또는 공휴일 |
| `is_chuseok_break` | 10/3 ~ 10/9 추석 7일 연휴 |

In [46]:
HOLIDAYS = {
    '2025-08-15': '광복절',
    '2025-10-03': '개천절',
    '2025-10-05': '추석 연휴', '2025-10-06': '추석', '2025-10-07': '추석 연휴',
    '2025-10-08': '대체공휴일(추석)',
    '2025-10-09': '한글날',
    '2025-12-25': '성탄절',
}

calendar = pd.DataFrame({'date': ALL_DAYS})
calendar['month'] = calendar['date'].dt.month
calendar['dow'] = calendar['date'].dt.dayofweek                        # 0=월요일 ... 6=일요일
calendar['dow_name'] = calendar['dow'].map(dict(enumerate('월화수목금토일')))
calendar['is_weekend'] = calendar['dow'] >= 5
calendar['holiday_name'] = calendar['date'].dt.strftime('%Y-%m-%d').map(HOLIDAYS).fillna('')
calendar['is_holiday'] = calendar['holiday_name'] != ''
calendar['is_offday'] = calendar['is_weekend'] | calendar['is_holiday']   # 쉬는 날 = 주말 또는 공휴일
calendar['is_chuseok_break'] = calendar['date'].between('2025-10-03', '2025-10-09')

print('쉬는 날:', calendar['is_offday'].sum(), '일 / 공휴일:', calendar['is_holiday'].sum(), '일')
display(calendar[calendar['is_holiday'] | calendar['is_chuseok_break']][['date', 'dow_name', 'holiday_name', 'is_offday']])
save(calendar, 'calendar')

쉬는 날: 59 일 / 공휴일: 8 일


,date,dow_name,holiday_name,is_offday
45,2025-08-15,금,광복절,True
94,2025-10-03,금,개천절,True
95,2025-10-04,토,,True
96,2025-10-05,일,추석 연휴,True
97,2025-10-06,월,추석,True
98,2025-10-07,화,추석 연휴,True
99,2025-10-08,수,대체공휴일(추석),True
100,2025-10-09,목,한글날,True
177,2025-12-25,목,성탄절,True


[저장] calendar.parquet  184행 × 9열, 0.0MB


## 13. 정리

### 13-1. 저장한 파일
이 노트북에서 `preprocessed/` 폴더에 저장한 파일 목록이다. 같은 이름으로 다시 저장한 파일은 마지막 것만 남는다.

In [47]:
pd.DataFrame(SAVED)

,파일,행,열,크기(MB)
0,card1.parquet,1044710,11,9.8
1,card2.parquet,1806800,11,16.4
2,wx_daily.parquet,30635,17,0.4
3,wx_hourly.parquet,692622,10,7.6
4,wx_warning_raw.parquet,4430,5,0.2
5,subway.parquet,822891,6,4.4
6,stores.parquet,163252,16,6.2
7,tour_dom_trend.parquet,360,7,0.0
8,tour_dom_home.parquet,2482,5,0.0
9,tour_for_trend.parquet,120,6,0.0


### 13-2. 발견한 문제와 처리
이 표는 `preprocessed/data_issues.csv`로도 저장된다.

In [48]:
issues = pd.DataFrame(ISSUES)
issues.to_csv(OUT / 'data_issues.csv', index=False, encoding='utf-8-sig')
pd.set_option('display.max_colwidth', None)    # 긴 글자도 잘리지 않게 전부 표시
issues

,데이터,문제,처리
0,카드1,"결제 5건 미만인 칸은 삭제돼 있음. 일별 행 수가 4,268~6,023개로 변동",184일 내내 나오는 칸에 is_balanced=True 표시 (3장)
1,카드1,법인 결제가 전체 금액의 15.9%,날씨 분석에서는 기본적으로 제외 (sex != '법인'). 02번에서 따로 확인
2,유동_time,"202512 파일에 완전히 같은 행이 92,545개 (모든 행이 두 번씩 들어 있음)",중복 제거
3,유동_wkdy,"202512 파일에 완전히 같은 행이 121,267개 (모든 행이 두 번씩 들어 있음)",중복 제거
4,유동,"경계에 걸친 격자가 이웃 시군구 코드로 들어 있음 {'age': 337, 'time': 330, 'wkdy': 641}",강남과 춘천 코드만 남김
5,유동_wkdy,평일엔 유동이 있는데 토요일과 일요일이 정확히 0인 격자가 5.8%,실제 0(업무지구)인지 마스킹인지 03번 노트북에서 확인
6,유동,age/time/wkdy 파일마다 격자 수가 다름 (같은 달이라도),세 표를 격자 단위로 합칠 때 주의. 03번에서 확인
7,기상_시간자료,"2025년 하반기 시간자료가 빠진 지점과 빠진 시간 수: {'강남': 33, '송파': 19, '남산': 35}",일자료는 184일 모두 있음. 시간 단위로 분석할 때만 주의
8,기상특보,"특보 종류, 행위(발표/해제 등), 구역이 한 문자열에 (1),(2)... 로 이어진 원문",04번에서 항목별로 쪼개서 파싱
9,지하철,"44개 파일 중 ['202402', '202502']는 cp949 인코딩 (README에는 utf-8-sig로 적혀 있음)",파일별 자동 판별
